# Notebook_A (Sinh viên A): dữ liệu, EDA, làm sạch, SQL, hình RQ1

**Đề tài**: Quỹ đạo xe không người lái 3 và 8 giây có bất định (Waymo Open Motion)

**Khung SDG**: SDG 11 và 9: an toàn xe tự hành; tiêu đề, keyword và abstract của bài dùng đúng cụm từ này.

**Vai trò**: Sinh viên A làm Bước 0 đến 3 và phần hình RQ1 của Bước 5, bàn giao `data/processed/feat.parquet` cùng `manifest.json` cho Sinh viên B (Notebook_B). Chạy tuần tự trong VS Code với extension Python và Jupyter, interpreter `.venv`. Mỗi bước có ô hướng dẫn *làm gì, vì sao, đọc thế nào* rồi đến ô code; mọi bảng lưu vào `report/` với tên `table_*.csv`, mọi hình `fig_*.png`.

**Quy tắc**: seed 42; không xoá dòng mà không ghi lý do; mọi prompt AI ghi vào Audit Log cá nhân; báo cáo slot 3 thứ Tư và thứ Bảy (12:30 đến 14:45) theo mẫu ở ô cuối.

## Vừa làm vừa hỏi AI và ghi Audit Log: cách làm nhẹ nhàng, không rối

**Tinh thần**: AI là bạn cùng làm, không phải người làm thay. Bạn được hỏi AI bất cứ lúc nào; điều duy nhất phải giữ là *kiểm tra một lần* trước khi dùng và *ghi lại 2 phút* nếu câu hỏi đó thay đổi việc bạn làm.

**Khi nào hỏi AI** (theo thứ tự):
1. Đọc ô hướng dẫn của bước đang làm (1 phút).
2. Thử tự làm 15 phút.
3. Bí quá 15 phút thì hỏi AI với mẫu: *"Tôi đang làm Bước X của dự án Y. Dữ liệu có cột A, B, C. Tôi muốn Z. Đây là code và lỗi: ... Hãy giải thích nguyên nhân và sửa, giữ nguyên tên biến."*
4. Chạy thử code AI đưa trên dữ liệu thật; so một con số bằng tay hoặc bằng cách thứ hai.
5. Nếu vẫn bí sau 30 phút, ghi lại lỗi và hỏi giảng viên ở kênh nhóm.

**Ghi Audit Log thế nào cho không áp lực**: chỉ ghi các prompt *đã thay đổi việc bạn làm* (thường 3 đến 5 prompt một tuần, 15 đến 20 cả kỳ). Mỗi dòng 5 ô, điền trong 2 phút ngay khi dùng, không để cuối tuần:

| Ngày, bước | Prompt (rút gọn) | AI trả lời gì (1 dòng) | Tôi kiểm tra hoặc sửa gì | Dùng vào đâu |
|---|---|---|---|---|
| 09/09, Bước 3 | viết truy vấn trung bình trượt 7 ngày theo trạm | đưa AVG OVER ROWS 6 PRECEDING | thiếu PARTITION BY site_num, thêm và thử 2 trạm | Q3 trong sql/queries.sql |

Cột thứ tư chính là *Human Delta* và là thứ hội đồng hỏi; ghi thật, kể cả khi AI đúng ("kiểm tra bằng 10 dòng tính tay, khớp"). Ba lần bạn phát hiện AI sai trong cả kỳ là đủ yêu cầu.

**Nhịp làm việc**: mỗi ngày 1,5 đến 2 giờ vào đúng bước của tuần, xong một ô thì commit; thứ Ba và thứ Sáu slot 1 báo cáo năm mục đúng tiến độ, dù kết quả chưa đẹp. Siêng và đúng nhịp quan trọng hơn giỏi: nhóm báo cáo đều 6 lần trong 3 tuần luôn có bài, nhóm im lặng đến tuần 3 thường không kịp.

## Bước 0: môi trường và cấu trúc thư mục

Chạy một lần trong Terminal của VS Code, không phải trong ô Python. Sau khi cài xong chọn interpreter `.venv` (Ctrl+Shift+P, *Python: Select Interpreter*).

```bash
# Step 1a: environment and project layout (run once)
python -m venv .venv && source .venv/bin/activate      # Windows: .venv\Scripts\activate
pip install pandas numpy duckdb pyarrow scikit-learn lightgbm xgboost catboost \
    statsmodels shap matplotlib seaborn ydata-profiling mapie jupyter
pip freeze > requirements.txt
mkdir -p data/raw data/processed notebooks sql src report
printf "data/raw\ndata/processed\n.venv\n*.parquet\n" > .gitignore
```

In [6]:
# Environment check and shared imports for the whole notebook
import pandas as pd, numpy as np, duckdb, matplotlib
import matplotlib.pyplot as plt, pathlib, warnings
warnings.filterwarnings("ignore")
for d in ["data/raw", "data/processed", "notebooks", "sql", "src", "report"]:
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
np.random.seed(42)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
print("pandas", pd.__version__, "| duckdb", duckdb.__version__)
# Paper figure style (docx 4.7 + visualisation guide 2.3, 2.5): drawn at real print size, 8-9 pt text,
# white background, pastel colours with ONE coral accent, dark-blue text/axes, no in-figure titles
FIG_W = 4.8  # LNCS text width (12.2 cm) in inches -> font sizes below are the printed sizes
C_TEXT, C_BASE, C_MID, C_ACCENT = "#1F3B57", "#9CC9E8", "#4F8FC0", "#FF7F66"
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 8, "axes.labelsize": 8, "xtick.labelsize": 8,
    "ytick.labelsize": 8, "legend.fontsize": 8, "text.color": C_TEXT, "axes.labelcolor": C_TEXT,
    "axes.edgecolor": C_TEXT, "xtick.color": C_TEXT, "ytick.color": C_TEXT, "axes.linewidth": 0.6,
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white", "axes.grid": False,
    "axes.spines.top": False, "axes.spines.right": False, "savefig.dpi": 300, "pdf.fonttype": 42})

pandas 2.3.3 | duckdb 1.5.5


## Vì sao sáu bước và mỗi bước giúp gì cho bài

| Bước | Mục đích | Lợi ích cho bài | Bằng chứng, câu hỏi |
|---|---|---|---|
| 1 Nạp, hiểu dữ liệu | biết quy mô, thời gian, đơn vị, mục tiêu | bảng mô tả dữ liệu, phạm vi hợp lệ | table_describe, profile; nền mọi RQ |
| 2 Làm sạch, ghép | dữ liệu tin được, có nguồn thứ hai | nhật ký làm sạch trả lời phản biện; đặc trưng ngoại sinh | cleaning_log, tỷ lệ khớp; RQ1, RQ2 |
| 3 SQL, EDA | trả lời câu hỏi mô tả bằng số có kiểm định; đặc trưng đúng thời điểm | bảng RQ1 có p-value; không rò rỉ; điểm SQL | table_q*, rq1a-c; RQ1 |
| 4 Chia dữ liệu | đánh giá công bằng | mốc naive cho biết cải thiện thật | table_features; RQ2 |
| 5 Trực quan hoá | thấy cấu trúc và bất thường; truyền đạt | hình có caption kết luận | fig_rq1_*; RQ1 |
| 6 Mô hình, RQ3 | so năm mô hình công bằng; đóng góp chính | bảng có độ lệch, tinh chỉnh có căn cứ, SHAP | table_rq2, rq3; RQ2, RQ3 |

**Vì sao phải chọn cột từ nhiều cột**: cột thừa làm mô hình học nhiễu, cột chứa giá trị tương lai làm kết quả đẹp giả, cột không có lúc vận hành làm mô hình vô dụng. Bốn bước giúp chọn đúng cột và khớp RQ: (1) tài liệu cơ quan cho biết ý nghĩa vật lý; (2) bảng thiếu và bảng tương quan trong EDA loại cột thiếu nhiều hoặc trùng; (3) SQL với LAG/LEAD tách quá khứ khỏi tương lai; (4) SHAP sau mô hình xác nhận cột thực sự đóng góp. Mỗi cột giữ lại phải nối về một RQ: cột mô tả cho RQ1, cột dự báo cho RQ2, cột nhóm hoặc điều kiện cho RQ3.

## Bước 1: hiểu bài toán và nạp dữ liệu

### 1.1. Bài toán và ba câu hỏi

Ghi lại ở đây ba câu hỏi nghiên cứu của đề cương (RQ1 mô tả, RQ2 mô hình, RQ3 câu hỏi thứ ba) và biến mục tiêu; mọi bảng và hình sau này phải nối về một trong ba câu hỏi.

**Làm gì**: tải file gốc từ trang cơ quan vào `data/raw/`, nạp bằng DuckDB (đọc thẳng file từ đĩa, chỉ lấy cột và dòng cần) rồi in số dòng, số cột, khoảng thời gian, số đơn vị.

**Kiểm tra**: `df.shape` khớp tài liệu cơ quan; khoảng thời gian đúng; số đơn vị đúng phạm vi đề cương.

### 1.2. Cấu hình dữ liệu và các hàm dùng chung

- **Core experiment**: WOMD **v1.2**.
- **Extension 1**: WOMD **v1.3.0** với traffic-light context.
- **Extension 2**: WOMD **v1.3.1** với traffic-light context.
- **Extension 3**: WOMD **v1.3.1 tf.Example** với `sdc_paths` (SDC-only).

Để thật sự dùng đủ 20 shard mà vẫn giữ đúng 1,200 trajectories, mỗi shard được lấy mẫu cân bằng bằng reservoir sampling. `n_agents` được tính từ **tất cả agent hợp lệ trong scene-step**; chỉ target trajectories bị giới hạn ở object type `vehicle`.


In [7]:
import glob, hashlib, json, os
import tensorflow as tf
from waymo_open_dataset.protos import scenario_pb2
N_SHARDS = 20
N_TRAJECTORIES = 1200
SEED = 42
# ---- docx 1a: one forecast origin per vehicle (end of second 1 = step 10) ----
ORIGIN_STEP = 10
COMMON_X = [
    "speed",
    "acc",
    "yaw_rate",
    "dspeed_10step",
    "acc_ma1s",
    "yaw_ma1s",
    "length",
    "n_agents",
]  # same 8 predictors for 3 s and 8 s
TRAFFIC_COLS = ["n_traffic_lights", "n_stop_lights", "n_caution_lights", "n_go_lights"]
# ---- docx 3.3: stationary vs moving ----
STOP_SPEED = 0.5  # m/s; below this the vehicle is treated as stationary
ZERO_DISP_EPS = 1e-6  # m; displacement treated as exactly zero
# ---- kinematic noise: |acc| up to ~156 m/s^2 and |yaw_rate| up to ~31 rad/s are not physical ----
CLIP_KINEMATICS = True
ACC_LIMIT = 10.0  # m/s^2 (about 1 g)
YAW_RATE_LIMIT = 2.0  # rad/s
# ---- reproducibility checks for the CORE sample (deterministic sampling, seed 42) ----
EXPECTED_CORE = {"rows": 109200, "units": 1200, "n3": 72363, "n8": 12894, "units3": 1199, "units8": 1175}
SAMPLING_LOGS = {}  # docx 5.4: filled by load_waymo_vehicle_sample
CLIP_LOG = {}  # filled by preprocess_vehicle_table
SCENARIO_DIRS = {
    "v1_2": "data/raw/womd_v1_2",
    "v1_3_0": "data/raw/womd_v1_3_0",
    "v1_3_1": "data/raw/womd_v1_3_1",
}
def get_scenario_files(version, n_shards=N_SHARDS):
    files = sorted(glob.glob(f"{SCENARIO_DIRS[version]}/training.tfrecord-*"))[:n_shards]
    assert len(files) == n_shards, f"{version}: expected {n_shards} shards, found {len(files)}"
    return files
def load_waymo_vehicle_sample(version, n_shards=N_SHARDS, total_trajectories=N_TRAJECTORIES, seed=SEED):
    """Balanced deterministic sample of complete 91-step vehicle trajectories across all shards."""
    files = get_scenario_files(version, n_shards)
    base_quota = total_trajectories // n_shards
    remainder = total_trajectories % n_shards
    all_track_rows = []
    for shard_i, fpath in enumerate(files):
        quota = base_quota + (1 if shard_i < remainder else 0)
        rng = np.random.default_rng(seed + shard_i)
        reservoir = []
        seen_eligible = 0
        n_scen_total = 0
        n_scen_91 = 0
        for rec in tf.data.TFRecordDataset(fpath, compression_type=""):
            sc = scenario_pb2.Scenario()
            sc.ParseFromString(rec.numpy())
            n_steps = len(sc.timestamps_seconds)
            n_scen_total += 1
            if n_steps != 91:
                continue
            n_scen_91 += 1
            density = np.zeros(n_steps, dtype=int)
            for tr_all in sc.tracks:
                for k, s in enumerate(tr_all.states[:n_steps]):
                    if s.valid:
                        density[k] += 1
            for tr in sc.tracks:
                if tr.object_type != 1:
                    continue
                if len(tr.states) != n_steps or not all(s.valid for s in tr.states):
                    continue
                track_rows = []
                for k, s in enumerate(tr.states):
                    track_rows.append(
                        (
                            sc.scenario_id,
                            tr.id,
                            k,
                            float(sc.timestamps_seconds[k]),
                            s.center_x,
                            s.center_y,
                            s.velocity_x,
                            s.velocity_y,
                            s.heading,
                            s.length,
                            s.width,
                            int(density[k]),
                        )
                    )
                seen_eligible += 1
                if len(reservoir) < quota:
                    reservoir.append(track_rows)
                else:
                    j = int(rng.integers(0, seen_eligible))
                    if j < quota:
                        reservoir[j] = track_rows
        assert (
            len(reservoir) == quota
        ), f"{version} shard {shard_i}: only {len(reservoir)} eligible trajectories"
        for track_rows in reservoir:
            all_track_rows.extend(track_rows)
        print(f"{version} shard {shard_i+1:02d}/{n_shards}: selected {len(reservoir)} trajectories")
        SAMPLING_LOGS.setdefault(version, []).append(
            {
                "version": version,
                "shard": shard_i + 1,
                "file": os.path.basename(fpath),
                "scenarios_scanned": n_scen_total,
                "scenarios_91_steps": n_scen_91,
                "eligible_vehicle_tracks": seen_eligible,
                "selected_tracks": len(reservoir),
                "selection_rate": len(reservoir) / max(seen_eligible, 1),
            }
        )
    df_raw = pd.DataFrame(
        all_track_rows,
        columns=[
            "scenario",
            "agent",
            "step",
            "ts",
            "x",
            "y",
            "vx",
            "vy",
            "heading",
            "length",
            "width",
            "n_agents",
        ],
    )
    n_units = df_raw[["scenario", "agent"]].drop_duplicates().shape[0]
    assert n_units == total_trajectories
    assert len(df_raw) == total_trajectories * 91
    print(f"{version}: {n_units} trajectories, {len(df_raw)} agent-step rows")
    return files, df_raw
def preprocess_vehicle_table(df_raw, label=None):
    """Idempotent kinematic preprocessing; always starts from a fresh raw copy."""
    df = df_raw.copy().sort_values(["scenario", "agent", "ts"]).reset_index(drop=True)
    df["unit"] = df["scenario"].astype(str) + "_" + df["agent"].astype(str)
    df["speed"] = np.hypot(df["vx"], df["vy"])
    df["dt"] = df.groupby("unit")["ts"].diff()
    bad_dt = df["dt"].notna() & (df["dt"] <= 0)
    assert not bad_dt.any(), "non-positive time differences found"
    df["acc"] = df.groupby("unit")["speed"].diff() / df["dt"]
    d_heading = df.groupby("unit")["heading"].diff()
    d_heading_wrapped = np.arctan2(np.sin(d_heading), np.cos(d_heading))
    df["yaw_rate"] = d_heading_wrapped / df["dt"]
    # Clip non-physical spikes caused by position/heading noise and dt jitter (logged for the cleaning table)
    if CLIP_KINEMATICS:
        n_acc = int((df["acc"].abs() > ACC_LIMIT).sum())
        n_yaw = int((df["yaw_rate"].abs() > YAW_RATE_LIMIT).sum())
        df["acc"] = df["acc"].clip(-ACC_LIMIT, ACC_LIMIT)
        df["yaw_rate"] = df["yaw_rate"].clip(-YAW_RATE_LIMIT, YAW_RATE_LIMIT)
        CLIP_LOG[label or "unlabeled"] = {
            "rows": len(df),
            "acc_clipped": n_acc,
            "yaw_rate_clipped": n_yaw,
            "acc_limit": ACC_LIMIT,
            "yaw_rate_limit": YAW_RATE_LIMIT,
        }
    counts = df.groupby("unit")["step"].count()
    assert (counts == 91).all(), "all sampled trajectories must have 91 steps"
    assert not df.duplicated(["unit", "step"]).any()
    return df
STOP_STATES = {1, 4, 7}
CAUTION_STATES = {2, 5, 8}
GO_STATES = {3, 6}
def build_traffic_context(files, needed_scenarios):
    needed_scenarios = set(needed_scenarios)
    rows, found = [], set()
    for fpath in files:
        for rec in tf.data.TFRecordDataset(fpath, compression_type=""):
            sc = scenario_pb2.Scenario()
            sc.ParseFromString(rec.numpy())
            if sc.scenario_id not in needed_scenarios:
                continue
            found.add(sc.scenario_id)
            for k, ts in enumerate(sc.timestamps_seconds):
                states = []
                if k < len(sc.dynamic_map_states):
                    states = [ls.state for ls in sc.dynamic_map_states[k].lane_states]
                rows.append(
                    (
                        sc.scenario_id,
                        k,
                        float(ts),
                        len(states),
                        sum(s in STOP_STATES for s in states),
                        sum(s in CAUTION_STATES for s in states),
                        sum(s in GO_STATES for s in states),
                    )
                )
    traffic = pd.DataFrame(
        rows,
        columns=[
            "scenario",
            "step",
            "traffic_ts",
            "n_traffic_lights",
            "n_stop_lights",
            "n_caution_lights",
            "n_go_lights",
        ],
    )
    missing = needed_scenarios - found
    assert not missing, f"traffic context missing for {len(missing)} selected scenarios"
    return traffic
def merge_traffic_context(df, traffic):
    out = df.merge(traffic, on=["scenario", "step"], how="left", validate="many_to_one")
    assert len(out) == len(df)
    assert out["traffic_ts"].notna().all(), "missing traffic rows after merge"
    time_diff = (out["ts"] - out["traffic_ts"]).abs()
    assert time_diff.max() < 1e-9, f"traffic timestamp mismatch: max={time_diff.max()}"
    return out.drop(columns=["traffic_ts"])
def build_feature_table(df, include_traffic=False, horizon_tol=0.05):
    """
    Build a leakage-safe feature table consistent with CORE v1.2.
    - dspeed_10step: speed difference over 10 samples
    - acc_ma1s / yaw_ma1s: trailing timestamp-based 1.0 s windows
    - y_disp_3s / y_disp_8s:
      retained only when actual elapsed time is within ±horizon_tol
    - future helper columns are not returned
    """
    c = duckdb.connect()
    c.register("wm_input", df)
    traffic_cols_inner = (
        ", n_traffic_lights, n_stop_lights, " "n_caution_lights, n_go_lights" if include_traffic else ""
    )
    traffic_cols_outer = (
        ", n_traffic_lights, n_stop_lights, " "n_caution_lights, n_go_lights" if include_traffic else ""
    )
    q = f"""
    WITH future_values AS (
        SELECT
            unit,
            scenario,
            step,
            ts,
            x,
            y,
            vx,
            vy,
            heading,
            length,
            width,
            n_agents,
            speed,
            acc,
            yaw_rate
            {traffic_cols_inner},
            speed - LAG(speed, 10) OVER (
                PARTITION BY unit
                ORDER BY step
            ) AS dspeed_10step,
            AVG(acc) OVER (
                PARTITION BY unit
                ORDER BY ts
                RANGE BETWEEN 1.0 PRECEDING AND CURRENT ROW
            ) AS acc_ma1s,
            AVG(yaw_rate) OVER (
                PARTITION BY unit
                ORDER BY ts
                RANGE BETWEEN 1.0 PRECEDING AND CURRENT ROW
            ) AS yaw_ma1s,
            LEAD(x, 30) OVER w AS x_lead30,
            LEAD(y, 30) OVER w AS y_lead30,
            LEAD(ts, 30) OVER w AS ts_lead30,
            LEAD(x, 80) OVER w AS x_lead80,
            LEAD(y, 80) OVER w AS y_lead80,
            LEAD(ts, 80) OVER w AS ts_lead80
        FROM wm_input
        WINDOW w AS (
            PARTITION BY unit
            ORDER BY step
        )
    )
    SELECT
        unit,
        scenario,
        step,
        ts,
        x,
        y,
        vx,
        vy,
        heading,
        length,
        width,
        n_agents,
        speed,
        acc,
        yaw_rate
        {traffic_cols_outer},
        dspeed_10step,
        acc_ma1s,
        yaw_ma1s,
        CASE
            WHEN ts_lead30 IS NOT NULL
             AND ABS((ts_lead30 - ts) - 3.0) <= {horizon_tol}
            THEN SQRT(
                POWER(x_lead30 - x, 2)
                + POWER(y_lead30 - y, 2)
            )
            ELSE NULL
        END AS y_disp_3s,
        CASE
            WHEN ts_lead80 IS NOT NULL
             AND ABS((ts_lead80 - ts) - 8.0) <= {horizon_tol}
            THEN SQRT(
                POWER(x_lead80 - x, 2)
                + POWER(y_lead80 - y, 2)
            )
            ELSE NULL
        END AS y_disp_8s
    FROM future_values
    """
    feat_out = c.execute(q).df()
    c.close()
    return feat_out
def build_origin_table(feat_df, label, extra_cols=(), origin_step=None):
    """docx 1a: keep ONE row per vehicle at the forecast origin (default step 10 = end of second 1).
    A row is kept only if BOTH targets (3 s and 8 s) and ALL predictors exist, so the 3 s and 8 s
    experiments use exactly the same vehicles, rows and features.
    Adds motion_state / zero-displacement flags (docx 3.3).
    """
    origin_step = ORIGIN_STEP if origin_step is None else origin_step
    feat_cols = COMMON_X + list(extra_cols)
    at_origin = feat_df[feat_df["step"] == origin_step].copy()
    has3 = at_origin["y_disp_3s"].notna()
    has8 = at_origin["y_disp_8s"].notna()
    hasx = at_origin[feat_cols].notna().all(axis=1)
    out = at_origin[has3 & has8 & hasx].copy()
    out["motion_state"] = np.where(out["speed"] < STOP_SPEED, "stationary", "moving")
    out["zero_disp_3s"] = (out["y_disp_3s"] <= ZERO_DISP_EPS).astype("int8")
    out["zero_disp_8s"] = (out["y_disp_8s"] <= ZERO_DISP_EPS).astype("int8")
    out = out.sort_values(["scenario", "unit"]).reset_index(drop=True)
    assert not out.duplicated("unit").any(), "origin table must have one row per vehicle"
    assert out[feat_cols + ["y_disp_3s", "y_disp_8s"]].notna().all().all()
    audit = pd.DataFrame(
        [
            {
                "table": label,
                "origin_step": origin_step,
                "origin_ts_median_s": float(at_origin["ts"].median()) if len(at_origin) else np.nan,
                "vehicles_at_origin": len(at_origin),
                "missing_3s_target": int((~has3).sum()),
                "missing_8s_target": int((~has8).sum()),
                "missing_predictor": int((~hasx).sum()),
                "final_rows": len(out),
                "final_vehicles": int(out["unit"].nunique()),
                "final_scenarios": int(out["scenario"].nunique()),
                "stationary_share": (
                    float((out["motion_state"] == "stationary").mean()) if len(out) else np.nan
                ),
                "zero_disp_3s_share": float(out["zero_disp_3s"].mean()) if len(out) else np.nan,
                "zero_disp_8s_share": float(out["zero_disp_8s"].mean()) if len(out) else np.nan,
            }
        ]
    )
    return out, audit

## CORE EXPERIMENT — WOMD v1.2

Các Bước 1–5, RQ1 và file bàn giao cho Notebook B đều dùng **v1.2**. Các version mới chỉ xuất hiện ở phần Extension sau Test A.


In [8]:
import inspect
src = inspect.getsource(build_feature_table)
print("dspeed_10step:", "dspeed_10step" in src)
print("timing validation:", "ts_lead30" in src and "horizon_tol" in src)
print("width included:", "width" in src)
print("old dspeed1s absent:", "AS dspeed1s" not in src)

dspeed_10step: True
timing validation: True
width included: True
old dspeed1s absent: True


In [9]:
# Step 1b (CORE): balanced 1,200-trajectory sample from all 20 WOMD v1.2 shards
files_12, df_raw_12 = load_waymo_vehicle_sample("v1_2")
print(df_raw_12.shape)
display(df_raw_12.head())

2026-10-11 20:17:16.013102: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:982] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-10-11 20:17:16.013906: W tensorflow/core/common_runtime/gpu/gpu_device.cc:1956] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...
2026-10-11 20:17:16.089684: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [1]
	 [[{{node Placeholder/_0}}]]


v1_2 shard 01/20: selected 60 trajectories


2026-10-11 20:17:20.437163: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [1]
	 [[{{node Placeholder/_0}}]]


v1_2 shard 02/20: selected 60 trajectories


2026-10-11 20:17:25.124275: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [1]
	 [[{{node Placeholder/_0}}]]


...

2026-10-11 20:18:36.057044: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [1]
	 [[{{node Placeholder/_0}}]]


v1_2 shard 19/20: selected 60 trajectories


2026-10-11 20:18:40.605452: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [1]
	 [[{{node Placeholder/_0}}]]


v1_2 shard 20/20: selected 60 trajectories
v1_2: 1200 trajectories, 109200 agent-step rows
(109200, 12)


,scenario,agent,step,ts,x,y,vx,vy,heading,length,width,n_agents
0,2cbe25abc7a2b90f,30,0,0.00000,8156.07666,4778.226074,0.0,0.0,6.913843,5.251483,2.207013,66
1,2cbe25abc7a2b90f,30,1,0.10002,8156.07666,4778.226074,0.0,0.0,6.913843,5.251483,2.207013,65
2,2cbe25abc7a2b90f,30,2,0.20001,8156.07666,4778.226074,0.0,0.0,6.913843,5.251483,2.207013,65
3,2cbe25abc7a2b90f,30,3,0.30004,8156.07666,4778.226074,0.0,0.0,6.913843,5.251483,2.207013,66
4,2cbe25abc7a2b90f,30,4,0.40003,8156.07666,4778.226074,0.0,0.0,6.913843,5.251483,2.207013,65


### 1.3. Quy tắc chọn 1,200 xe (góp ý 5.4)

Mỗi shard trong 20 shard WOMD v1.2 đóng góp đúng 60 xe. Một xe đủ điều kiện khi: là `vehicle`, nằm trong cảnh có đủ 91 bước, và cả 91 trạng thái đều hợp lệ. Trong mỗi shard, 60 xe được chọn ngẫu nhiên đều (reservoir sampling, seed 42 + số thứ tự shard) trong số các xe đủ điều kiện. Con số 1,200 là ngân sách tính toán cố định (20 × 60). Ô dưới lưu số liệu để viết câu này vào bài.

In [10]:
# Step 1c (docx 5.4): document the sampling rule with numbers
sampling_12 = pd.DataFrame(SAMPLING_LOGS["v1_2"])
sampling_12.to_csv("report/table_sampling_v1_2.csv", index=False)
display(sampling_12)
veh_per_scen = df_raw_12.groupby("scenario")["agent"].nunique()
sampling_summary = {
    "shards": len(sampling_12),
    "scenarios_scanned": int(sampling_12["scenarios_scanned"].sum()),
    "scenarios_91_steps": int(sampling_12["scenarios_91_steps"].sum()),
    "eligible_vehicle_tracks": int(sampling_12["eligible_vehicle_tracks"].sum()),
    "selected_tracks": int(sampling_12["selected_tracks"].sum()),
    "selected_scenarios": int(df_raw_12["scenario"].nunique()),
    "vehicles_per_selected_scenario_mean": round(float(veh_per_scen.mean()), 3),
    "vehicles_per_selected_scenario_max": int(veh_per_scen.max()),
    "scenarios_with_1_selected_vehicle": int((veh_per_scen == 1).sum()),
}
pd.Series(sampling_summary).to_csv("report/table_sampling_summary_v1_2.csv", header=["value"])
print(pd.Series(sampling_summary))
print(
    f"\nSentence for the paper: From each of the {sampling_summary['shards']} WOMD v1.2 training shards we drew "
    f"{N_TRAJECTORIES // N_SHARDS} vehicle tracks uniformly at random (reservoir sampling, seed {SEED}) among the "
    f"{sampling_summary['eligible_vehicle_tracks']:,} vehicle tracks that are valid at all 91 steps, giving "
    f"{sampling_summary['selected_tracks']:,} tracks from {sampling_summary['selected_scenarios']:,} scenarios "
    f"(on average {sampling_summary['vehicles_per_selected_scenario_mean']} selected vehicles per scenario)."
)

,version,shard,file,scenarios_scanned,scenarios_91_steps,eligible_vehicle_tracks,selected_tracks,selection_rate
0,v1_2,1,training.tfrecord-00000-of-01000,492,492,7703,60,0.007789
1,v1_2,2,training.tfrecord-00001-of-01000,514,514,8249,60,0.007274
2,v1_2,3,training.tfrecord-00002-of-01000,516,516,7837,60,0.007656
3,v1_2,4,training.tfrecord-00003-of-01000,475,475,7811,60,0.007681
4,v1_2,5,training.tfrecord-00004-of-01000,522,522,8383,60,0.007157
5,v1_2,6,training.tfrecord-00005-of-01000,488,488,7929,60,0.007567
6,v1_2,7,training.tfrecord-00006-of-01000,558,558,8848,60,0.006781
7,v1_2,8,training.tfrecord-00007-of-01000,439,439,7087,60,0.008466
8,v1_2,9,training.tfrecord-00008-of-01000,497,497,8325,60,0.007207
9,v1_2,10,training.tfrecord-00009-of-01000,502,502,8008,60,0.007493


shards                                     20.000
scenarios_scanned                        9892.000
scenarios_91_steps                       9892.000
eligible_vehicle_tracks                157297.000
selected_tracks                          1200.000
selected_scenarios                       1073.000
vehicles_per_selected_scenario_mean         1.118
vehicles_per_selected_scenario_max          4.000
scenarios_with_1_selected_vehicle         965.000
dtype: float64

Sentence for the paper: From each of the 20 WOMD v1.2 training shards we drew 60 vehicle tracks uniformly at random (reservoir sampling, seed 42) among the 157,297 vehicle tracks that are valid at all 91 steps, giving 1,200 tracks from 1,073 scenarios (on average 1.118 selected vehicles per scenario).


## Bước 2: hiểu dữ liệu và làm sạch, ghép nguồn thứ hai

**Làm gì**: chuẩn tên cột; bỏ trùng theo khoá; bỏ thiếu ở biến mục tiêu; loại giá trị ngoài khoảng vật lý theo tài liệu cơ quan; dựng khoá thời gian đầy đủ và nội suy ngắn; ghép nguồn thứ hai; lưu parquet.

**Kiểm tra**: số dòng sau merge không tăng; tỷ lệ khớp trên 90%; ghi số dòng bỏ ở mỗi bước vào nhật ký làm sạch (ô sau).

In [11]:
# Step 2 (CORE): kinematics and clean vehicle table
# Always restart from df_raw_12, so re-running this cell cannot create n_agents_x / n_agents_y.
df = preprocess_vehicle_table(df_raw_12, label="v1_2")
df.to_parquet("data/processed/womd_v1_2.parquet", index=False)
print(df.shape, df["unit"].nunique())
print(df[["step", "ts", "dt", "speed", "acc", "yaw_rate", "n_agents"]].head(15))
print("\nTime-step statistics:")
print(df["dt"].dropna().describe())
print("\nKinematic clipping:", CLIP_LOG.get("v1_2"))

(109200, 17) 1200
    step       ts       dt  speed  acc  yaw_rate  n_agents
0      0  0.00000      NaN    0.0  NaN       NaN        12
1      1  0.09997  0.09997    0.0  0.0       0.0        12
2      2  0.19993  0.09996    0.0  0.0       0.0        12
3      3  0.29993  0.10000    0.0  0.0       0.0        12
4      4  0.39990  0.09997    0.0  0.0       0.0        13
5      5  0.49990  0.10000    0.0  0.0       0.0        13
6      6  0.59990  0.10000    0.0  0.0       0.0        13
7      7  0.69991  0.10001    0.0  0.0       0.0        13
8      8  0.79991  0.10000    0.0  0.0       0.0        13
9      9  0.89995  0.10004    0.0  0.0       0.0        13
10    10  0.99995  0.10000    0.0  0.0       0.0        13
11    11  1.09999  0.10004    0.0  0.0       0.0        13
12    12  1.19999  0.10000    0.0  0.0       0.0        13
13    13  1.29999  0.10000    0.0  0.0       0.0        13
14    14  1.39999  0.10000    0.0  0.0       0.0        13

Time-step statistics:
count    108000

### 2.0. Hồ sơ tự động

Tạo profile trên bảng core v1.2 sau preprocessing để kiểm tra phân phối, missing và outlier. Đây là bước mô tả, không thay đổi dữ liệu.


In [12]:
# Validate sampling interval before constructing 3s / 8s targets
dt_valid = df["dt"].dropna()
print("Total dt intervals:", len(dt_valid))
print("dt approximately 0.1s:", np.isclose(dt_valid, 0.1, atol=0.005).sum())
print("Outside 0.095–0.105s:", ((dt_valid < 0.095) | (dt_valid > 0.105)).sum())
print("\nLargest dt values:")
display(
    df.loc[df["dt"].notna(), ["scenario", "unit", "step", "ts", "dt"]]
    .sort_values("dt", ascending=False)
    .head(20)
)
print("\nSmallest dt values:")
display(df.loc[df["dt"].notna(), ["scenario", "unit", "step", "ts", "dt"]].sort_values("dt").head(20))

Total dt intervals: 108000
dt approximately 0.1s: 107945
Outside 0.095–0.105s: 55

Largest dt values:


,scenario,unit,step,ts,dt
36109,5cb8a511548cd234,5cb8a511548cd234_31,73,7.80024,0.30002
13812,2a59ed5a3b80f350,2a59ed5a3b80f350_1280,71,7.49969,0.29948
13827,2a59ed5a3b80f350,2a59ed5a3b80f350_1280,86,9.09873,0.20008
76278,b766dc0f5bad60e7,b766dc0f5bad60e7_2254,20,2.10003,0.20008
76369,b766dc0f5bad60e7,b766dc0f5bad60e7_2270,20,2.10003,0.20008
99000,ea79bc45404eb541,ea79bc45404eb541_2366,83,8.40025,0.20006
13778,2a59ed5a3b80f350,2a59ed5a3b80f350_1280,37,3.80006,0.20005
57887,90953e191056ce17,90953e191056ce17_63,11,1.20004,0.20004
4374,194c7292ea2aebfd,194c7292ea2aebfd_3189,6,0.70005,0.20004
75263,b50a5f78ac11dd6e,b50a5f78ac11dd6e_688,6,0.80018,0.20004



Smallest dt values:


,scenario,unit,step,ts,dt
5734,1bfea62214cade63,1bfea62214cade63_13,1,0.05920,0.05920
46047,7166addd759f4176,7166addd759f4176_2686,1,0.07350,0.07350
56421,8d6c8b111f0a6a31,8d6c8b111f0a6a31_0,1,0.07418,0.07418
96916,e64f3709bf45690,e64f3709bf45690_1778,1,0.08939,0.08939
103923,f4d028153147147,f4d028153147147_982,1,0.09844,0.09844
45956,70c378e5c156b189,70c378e5c156b189_1424,1,0.09847,0.09847
26825,42c09f83c149dac9,42c09f83c149dac9_79,71,7.08251,0.09889
36002,5c59da81f1b56947,5c59da81f1b56947_1942,57,5.67275,0.09890
36006,5c59da81f1b56947,5c59da81f1b56947_1942,61,6.06854,0.09892
36000,5c59da81f1b56947,5c59da81f1b56947_1942,55,5.47492,0.09892


In [13]:
# Validate actual elapsed time for 30-step and 80-step horizons
tmp = df.sort_values(["unit", "step"]).copy()
tmp["ts_plus_30"] = tmp.groupby("unit")["ts"].shift(-30)
tmp["ts_plus_80"] = tmp.groupby("unit")["ts"].shift(-80)
tmp["elapsed_3s"] = tmp["ts_plus_30"] - tmp["ts"]
tmp["elapsed_8s"] = tmp["ts_plus_80"] - tmp["ts"]
valid_3 = tmp["elapsed_3s"].dropna()
valid_8 = tmp["elapsed_8s"].dropna()
print("3s horizon")
print(valid_3.describe())
print("Outside 2.95–3.05 s:", ((valid_3 < 2.95) | (valid_3 > 3.05)).sum())
print("\n8s horizon")
print(valid_8.describe())
print("Outside 7.95–8.05 s:", ((valid_8 < 7.95) | (valid_8 > 8.05)).sum())
print("\nLargest 3s deviations:")
display(
    tmp.loc[tmp["elapsed_3s"].notna(), ["scenario", "unit", "step", "ts", "elapsed_3s"]]
    .assign(error=lambda x: (x["elapsed_3s"] - 3.0).abs())
    .sort_values("error", ascending=False)
    .head(15)
)
print("\nLargest 8s deviations:")
display(
    tmp.loc[tmp["elapsed_8s"].notna(), ["scenario", "unit", "step", "ts", "elapsed_8s"]]
    .assign(error=lambda x: (x["elapsed_8s"] - 8.0).abs())
    .sort_values("error", ascending=False)
    .head(15)
)

3s horizon
count    73200.000000
mean         3.001281
std          0.015392
min          2.959200
25%          2.999890
50%          3.000000
75%          3.000110
max          3.399970
Name: elapsed_3s, dtype: float64
Outside 2.95–3.05 s: 837

8s horizon
count    13200.000000
mean         8.003413
std          0.032787
min          7.950030
25%          7.999830
50%          8.000000
75%          8.000150
max          8.600050
Name: elapsed_8s, dtype: float64
Outside 7.95–8.05 s: 306

Largest 3s deviations:


,scenario,unit,step,ts,elapsed_3s,error
36084,5cb8a511548cd234,5cb8a511548cd234_31,48,5.00018,3.39997,0.39997
36083,5cb8a511548cd234,5cb8a511548cd234_31,47,4.90019,3.39997,0.39997
36085,5cb8a511548cd234,5cb8a511548cd234_31,49,5.10014,3.39996,0.39996
36088,5cb8a511548cd234,5cb8a511548cd234_31,52,5.40013,3.39989,0.39989
36087,5cb8a511548cd234,5cb8a511548cd234_31,51,5.30014,3.39989,0.39989
36086,5cb8a511548cd234,5cb8a511548cd234_31,50,5.20016,3.39989,0.39989
13801,2a59ed5a3b80f350,2a59ed5a3b80f350_1280,60,6.10047,3.39857,0.39857
13800,2a59ed5a3b80f350,2a59ed5a3b80f350_1280,59,6.00040,3.39854,0.39854
13798,2a59ed5a3b80f350,2a59ed5a3b80f350_1280,57,5.80033,3.39846,0.39846
13799,2a59ed5a3b80f350,2a59ed5a3b80f350_1280,58,5.90039,3.39846,0.39846



Largest 8s deviations:


,scenario,unit,step,ts,elapsed_8s,error
36036,5cb8a511548cd234,5cb8a511548cd234_31,0,0.00000,8.60005,0.60005
36037,5cb8a511548cd234,5cb8a511548cd234_31,1,0.09998,8.60005,0.60005
36038,5cb8a511548cd234,5cb8a511548cd234_31,2,0.20001,8.60001,0.60001
36039,5cb8a511548cd234,5cb8a511548cd234_31,3,0.30003,8.59996,0.59996
36046,5cb8a511548cd234,5cb8a511548cd234_31,10,1.00009,8.59991,0.59991
36043,5cb8a511548cd234,5cb8a511548cd234_31,7,0.70007,8.59991,0.59991
36045,5cb8a511548cd234,5cb8a511548cd234_31,9,0.90011,8.59990,0.59990
36044,5cb8a511548cd234,5cb8a511548cd234_31,8,0.80010,8.59990,0.59990
36040,5cb8a511548cd234,5cb8a511548cd234_31,4,0.40005,8.59989,0.59989
36041,5cb8a511548cd234,5cb8a511548cd234_31,5,0.50007,8.59989,0.59989


In [14]:
from ydata_profiling import ProfileReport
sample = df.sample(min(20000, len(df)), random_state=42)
ProfileReport(sample, minimal=True).to_file("report/profile.html")
df.describe(include="all").T.to_csv("report/table_describe_raw.csv")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 17/17 [00:00<00:00, 593.45it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

### 2.1. Nhật ký làm sạch (bảng đưa vào bài)

Mỗi bước: tên bước, số dòng trước, số dòng sau, số bỏ, lý do có tài liệu.

In [15]:
# Step 2.1 (CORE): preprocessing / validation audit log
n_raw = len(df_raw_12)
n_processed = len(df)
duplicate_count = int(df.duplicated(["unit", "step"]).sum())
missing_speed_count = int(df["speed"].isna().sum())
bad_dt_count = int((df["dt"].notna() & (df["dt"] <= 0)).sum())
steps_per_unit = df.groupby("unit")["step"].size()
bad_unit_count = int((steps_per_unit != 91).sum())
# Core integrity checks
assert n_processed == n_raw, "Preprocessing unexpectedly changed row count"
assert duplicate_count == 0, "Duplicate unit-step rows found"
assert missing_speed_count == 0, "Missing speed values found"
assert bad_dt_count == 0, "Non-positive dt found"
assert bad_unit_count == 0, "Some trajectories do not contain 91 steps"
cleaning_log = pd.DataFrame(
    [
        {
            "step": "core sampled input",
            "rows_before": n_raw,
            "rows_after": n_raw,
            "dropped": 0,
            "reason": "loader already keeps only complete 91-step valid vehicle trajectories",
        },
        {
            "step": "derive kinematic features",
            "rows_before": n_raw,
            "rows_after": n_processed,
            "dropped": n_raw - n_processed,
            "reason": "derive unit, speed, dt, acceleration and yaw rate; no row removal",
        },
        {
            "step": "clip non-physical kinematics",
            "rows_before": n_processed,
            "rows_after": n_processed,
            "dropped": 0,
            "reason": (
                f"values clipped, not dropped: acc beyond ±{ACC_LIMIT} m/s^2 in "
                f"{CLIP_LOG.get('v1_2', {}).get('acc_clipped', 0)} rows; yaw rate beyond ±{YAW_RATE_LIMIT} rad/s in "
                f"{CLIP_LOG.get('v1_2', {}).get('yaw_rate_clipped', 0)} rows"
                if CLIP_KINEMATICS
                else "clipping disabled"
            ),
        },
        {
            "step": "validate duplicate unit-step",
            "rows_before": n_processed,
            "rows_after": n_processed,
            "dropped": 0,
            "reason": f"{duplicate_count} duplicate unit-step rows found",
        },
        {
            "step": "validate missing speed",
            "rows_before": n_processed,
            "rows_after": n_processed,
            "dropped": 0,
            "reason": f"{missing_speed_count} missing speed values found",
        },
        {
            "step": "validate temporal integrity",
            "rows_before": n_processed,
            "rows_after": n_processed,
            "dropped": 0,
            "reason": f"{bad_dt_count} non-positive dt; {bad_unit_count} units without 91 steps",
        },
    ]
)
cleaning_log.to_csv("report/table_cleaning_log.csv", index=False)
print(cleaning_log.to_string(index=False))

                        step  rows_before  rows_after  dropped                                                                                                reason
          core sampled input       109200      109200        0                                 loader already keeps only complete 91-step valid vehicle trajectories
   derive kinematic features       109200      109200        0                                     derive unit, speed, dt, acceleration and yaw rate; no row removal
clip non-physical kinematics       109200      109200        0 values clipped, not dropped: acc beyond ±10.0 m/s^2 in 574 rows; yaw rate beyond ±2.0 rad/s in 3 rows
validate duplicate unit-step       109200      109200        0                                                                      0 duplicate unit-step rows found
      validate missing speed       109200      109200        0                                                                          0 missing speed values found
 validate 

### 1.2. EDA phần 1: cấu trúc bảng (info, describe) và cách đọc

**Làm gì**: `info()` cho biết kiểu dữ liệu và số ô không thiếu của từng cột; `describe()` cho thống kê năm số và trung bình; với cột phân loại xem số giá trị khác nhau và giá trị phổ biến nhất.

**Đọc thế nào**: cột số mà `dtype` là object nghĩa là có ký tự lạ cần ép kiểu; `max` bất thường (ví dụ 999, -9999) là mã thiếu của cơ quan; `std` bằng 0 là cột hằng, bỏ. Ghi lại các phát hiện này vào bảng `report/table_eda_notes.csv` để đưa vào mục Data của bài.

In [16]:
eda = df.copy()
import io
buf = io.StringIO()
eda.info(buf=buf)
print(buf.getvalue()[:3000])
desc_num = eda.describe().T
desc_num["skew"] = eda.select_dtypes("number").skew()
desc_num.round(3).to_csv("report/table_describe_numeric.csv")
print(desc_num.round(3).head(15))
desc_cat = eda.select_dtypes(exclude="number").describe().T
desc_cat.to_csv("report/table_describe_categorical.csv")
print(desc_cat.head(15))
notes = []
for c in eda.select_dtypes("number").columns:
    s = eda[c]
    if s.std() == 0:
        notes.append([c, "constant column, drop"])
    if s.max() in (999, 9999, 99999, -9999):
        notes.append([c, "agency missing code, replace with NaN"])
    if abs(s.skew()) > 2:
        notes.append([c, "strongly skewed, consider log or tree models"])
pd.DataFrame(notes, columns=["column", "note"]).to_csv("report/table_eda_notes.csv", index=False)
print(notes)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 109200 entries, 0 to 109199
Data columns (total 17 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   scenario  109200 non-null  object 
 1   agent     109200 non-null  int64  
 2   step      109200 non-null  int64  
 3   ts        109200 non-null  float64
 4   x         109200 non-null  float64
 5   y         109200 non-null  float64
 6   vx        109200 non-null  float64
 7   vy        109200 non-null  float64
 8   heading   109200 non-null  float64
 9   length    109200 non-null  float64
 10  width     109200 non-null  float64
 11  n_agents  109200 non-null  int64  
 12  unit      109200 non-null  object 
 13  speed     109200 non-null  float64
 14  dt        108000 non-null  float64
 15  acc       108000 non-null  float64
 16  yaw_rate  108000 non-null  float64
dtypes: float64(12), int64(3), object(2)
memory usage: 14.2+ MB

             count      mean       std        min       25%     

### 1.3. EDA phần 2: thiếu dữ liệu, mẫu thiếu và quyết định xử lý

**Cách xem**: tỷ lệ thiếu từng cột; thiếu theo thời gian (có tháng nào mất hẳn không); thiếu theo đơn vị (trạm, pin, bang); thiếu có đi cùng nhau không (ma trận đồng thiếu).

**Quyết định xử lý** (ghi vào bảng và vào bài):
- Thiếu ở **biến mục tiêu**: bỏ dòng, không bao giờ điền.
- Thiếu **ngắn** trong chuỗi thời gian (1-3 bước): nội suy tuyến tính theo từng đơn vị.
- Thiếu **dài** hoặc cả tháng: bỏ đoạn đó của đơn vị, ghi lý do.
- Cột thiếu trên 40%: bỏ cột hoặc chỉ giữ cờ có/không.
- Cột phân loại thiếu: gộp vào lớp `unknown`.
- Với mô hình cây (LightGBM, XGBoost) thiếu trong đặc trưng có thể để nguyên, mô hình tự xử lý; với ridge phải điền (median) sau khi chia train/test.

In [17]:
# EDA 1.3 (CORE): missingness audit with structural first-step NaNs handled explicitly
miss = eda.isna().mean().sort_values(ascending=False).rename("missing_rate").to_frame()
miss["missing_count"] = eda.isna().sum()
# Explicit decision logic
decisions = []
for c in miss.index:
    rate = miss.loc[c, "missing_rate"]
    count = miss.loc[c, "missing_count"]
    if c in ["dt", "acc", "yaw_rate"]:
        decisions.append("keep structural NaN at first step of each trajectory; do not impute here")
    elif rate == 0:
        decisions.append("keep")
    elif rate < 0.05:
        decisions.append("review; impute only if justified downstream")
    elif rate < 0.40:
        decisions.append("drop or keep missingness flag")
    else:
        decisions.append("drop column")
miss["decision"] = decisions
miss.round(4).to_csv("report/table_missing.csv")
print(miss.head(17))
# Check structural missingness by trajectory step
print("\nMissing counts for derived temporal columns:")
print(eda[["dt", "acc", "yaw_rate"]].isna().sum())
print("\nMissing counts by step for dt/acc/yaw_rate:")
structural_by_step = eda.groupby("step")[["dt", "acc", "yaw_rate"]].apply(lambda x: x.isna().sum())
print(structural_by_step.head(5))
# Target missingness
target_missing_by_step = eda.groupby("step")["speed"].apply(lambda s: s.isna().mean())
target_missing_by_unit = (
    eda.groupby("unit")["speed"].apply(lambda s: s.isna().mean()).sort_values(ascending=False)
)
print("\nTarget missing by step (max):", target_missing_by_step.max())
print("Target missing by unit (max):", target_missing_by_unit.max())
# Co-missingness
co = eda.isna().astype(int)
co = co.loc[:, co.sum() > 0]
print("\nColumns with missing values:")
print(co.columns.tolist())
if co.shape[1] > 1:
    print("\nCo-missingness correlation:")
    print(co.corr().round(2))
# Missing data are three columns at one step only, so a sentence replaces the old bar chart (guide 1.3)
miss_cols = miss[miss["missing_rate"] > 0]
miss_sentence = (f"Only {', '.join(miss_cols.index)} are missing, each in {int(miss_cols['missing_count'].iloc[0]):,} rows "
    f"({100 * miss_cols['missing_rate'].iloc[0]:.1f}%), all at the first step of every trajectory; no other column has missing values.")
open("report/missing_sentence.txt", "w", encoding="utf-8").write(miss_sentence)
print("\n" + miss_sentence)

          missing_rate  missing_count                                           decision
yaw_rate      0.010989           1200  keep structural NaN at first step of each traj...
acc           0.010989           1200  keep structural NaN at first step of each traj...
dt            0.010989           1200  keep structural NaN at first step of each traj...
length        0.000000              0                                               keep
speed         0.000000              0                                               keep
unit          0.000000              0                                               keep
n_agents      0.000000              0                                               keep
width         0.000000              0                                               keep
scenario      0.000000              0                                               keep
agent         0.000000              0                                               keep
vy            0.00000

### 1.4. EDA phần 3: chọn cột và thống kê theo nhóm, theo thời gian

**Chọn cột** theo ba tiêu chí: (1) có ý nghĩa vật lý hoặc nghiệp vụ với mục tiêu; (2) có sẵn tại thời điểm dự báo (không phải giá trị tương lai); (3) không trùng lặp với cột khác (tương quan trên 0,95 thì giữ một). Bảng `table_columns.csv` ghi cột nào giữ, cột nào bỏ và vì sao; bảng này là một phần của mục Data trong bài.

**Thống kê**: trung bình, trung vị, phân vị 5 và 95 của mục tiêu theo nhóm và theo thời kỳ; đây là bảng đầu tiên trả lời RQ1 trước khi làm SQL.

In [18]:
print("Target missing by step (max):", target_missing_by_step.max())
print("Target missing by unit (max):", target_missing_by_unit.max())
print("\nStructural missingness first 5 steps:")
print(structural_by_step.head())

Target missing by step (max): 0.0
Target missing by unit (max): 0.0

Structural missingness first 5 steps:
        dt   acc  yaw_rate
step                      
0     1200  1200      1200
1        0     0         0
2        0     0         0
3        0     0         0
4        0     0         0


In [19]:
# EDA 1.4 (CORE): current-state feature audit before future targets are created
num_cols = [c for c in ["speed", "acc", "yaw_rate", "n_agents"] if c in eda.columns]
# Correlation among CURRENT-STATE variables
corr = eda[num_cols].corr()
corr.round(3).to_csv("report/table_corr.csv")
print("Current-state correlation matrix:")
print(corr.round(3))
# Column role audit
cols = []
for c in eda.columns:
    if c in ["scenario", "agent", "unit"]:
        cols.append([c, "identifier / grouping key", "keep for grouping; not a direct model feature"])
    elif c in ["step", "ts"]:
        cols.append([c, "temporal key", "keep for ordering and target construction"])
    elif c in ["x", "y"]:
        cols.append([c, "current vehicle position", "keep for future displacement target construction"])
    elif c in ["vx", "vy", "heading", "length", "width"]:
        cols.append([c, "current-state vehicle attribute", "keep for review"])
    elif c == "speed":
        cols.append([c, "current-state speed; NOT the final forecasting target", "keep"])
    elif c in ["dt", "acc", "yaw_rate"]:
        cols.append(
            [c, "derived current/past kinematic variable", "keep; structural NaN at first trajectory step"]
        )
    elif c == "n_agents":
        cols.append([c, "current scene density", "keep"])
    else:
        cols.append([c, "other", "review"])
column_audit = pd.DataFrame(cols, columns=["column", "reason", "decision"])
column_audit.to_csv("report/table_columns.csv", index=False)
print("\nColumn audit:")
print(column_audit.to_string(index=False))
# Important:
# y_disp_3s and y_disp_8s do not exist yet.
# Final target statistics will be produced AFTER Step 3 target validation.
assert "y_disp_3s" not in eda.columns
assert "y_disp_8s" not in eda.columns
print("\nOK: future targets have not been created yet. " "Target statistics are deferred until Step 3.")

Current-state correlation matrix:
          speed    acc  yaw_rate  n_agents
speed     1.000  0.036     0.027    -0.307
acc       0.036  1.000     0.001    -0.007
yaw_rate  0.027  0.001     1.000    -0.005
n_agents -0.307 -0.007    -0.005     1.000

Column audit:
  column                                                reason                                         decision
scenario                             identifier / grouping key    keep for grouping; not a direct model feature
   agent                             identifier / grouping key    keep for grouping; not a direct model feature
    step                                          temporal key        keep for ordering and target construction
      ts                                          temporal key        keep for ordering and target construction
       x                              current vehicle position keep for future displacement target construction
       y                              current vehicle position k

## Bước 3: phân tích bằng SQL, trả lời RQ1 và tạo đặc trưng

**Làm gì**: đưa parquet vào DuckDB; Q1 và Q2 tổng hợp trả lời RQ1 (GROUP BY, window, CTE); Q3 tạo đặc trưng trễ, trung bình trượt và mục tiêu tương lai bằng LAG, LEAD, AVG OVER; Q4 kiểm tra chất lượng. Lưu toàn bộ vào `sql/queries.sql`; mỗi truy vấn SELECT được lưu thành `report/table_q*.csv`.

**Kiểm tra**: chọn một đơn vị và 10 dòng, tính tay trung bình trượt rồi so với SQL; không đặc trưng nào dùng giá trị sau thời điểm dự báo.

In [20]:
# Step 2.5 (CORE): verify actual elapsed time for LEAD 30 / LEAD 80
horizon_check = df[["unit", "step", "ts"]].sort_values(["unit", "step"]).copy()
g = horizon_check.groupby("unit", sort=False)
# Future timestamps corresponding to SQL LEAD(..., 30) / LEAD(..., 80)
horizon_check["ts_lead30"] = g["ts"].shift(-30)
horizon_check["ts_lead80"] = g["ts"].shift(-80)
horizon_check["elapsed_30"] = horizon_check["ts_lead30"] - horizon_check["ts"]
horizon_check["elapsed_80"] = horizon_check["ts_lead80"] - horizon_check["ts"]
# Availability check
n30 = horizon_check["elapsed_30"].notna().sum()
n80 = horizon_check["elapsed_80"].notna().sum()
expected30 = N_TRAJECTORIES * (91 - 30)
expected80 = N_TRAJECTORIES * (91 - 80)
print("LEAD 30 availability:")
print(" expected:", expected30)
print(" actual  :", n30)
print("\nLEAD 80 availability:")
print(" expected:", expected80)
print(" actual  :", n80)
assert n30 == expected30
assert n80 == expected80
# Elapsed-time statistics
for col, target in [("elapsed_30", 3.0), ("elapsed_80", 8.0)]:
    s = horizon_check[col].dropna()
    print("\n" + "=" * 60)
    print(f"{col}  | nominal target = {target:.1f} s")
    print("=" * 60)
    print(s.describe())
    print("\nAbsolute timing error:")
    print((s - target).abs().describe())
    print("\nWithin tolerance:")
    print("±0.01 s:", int(((s - target).abs() <= 0.01).sum()), "/", len(s))
    print("±0.05 s:", int(((s - target).abs() <= 0.05).sum()), "/", len(s))
    print("outside ±0.05 s:", int(((s - target).abs() > 0.05).sum()))
    print("\nRounded elapsed-time frequencies:")
    print(s.round(2).value_counts().sort_index().to_string())
    print("\nLargest elapsed times:")
    print(s.nlargest(10).to_list())

LEAD 30 availability:
 expected: 73200
 actual  : 73200

LEAD 80 availability:
 expected: 13200
 actual  : 13200

elapsed_30  | nominal target = 3.0 s
count    73200.000000
mean         3.001281
std          0.015392
min          2.959200
25%          2.999890
50%          3.000000
75%          3.000110
max          3.399970
Name: elapsed_30, dtype: float64

Absolute timing error:
count    73200.000000
mean         0.002548
std          0.015233
min          0.000000
25%          0.000040
50%          0.000110
75%          0.000320
max          0.399970
Name: elapsed_30, dtype: float64

Within tolerance:
±0.01 s: 69323 / 73200
±0.05 s: 72363 / 73200
outside ±0.05 s: 837

Rounded elapsed-time frequencies:
elapsed_30
2.96        1
2.97       87
2.98      938
2.99     1668
3.00    67640
3.01     1336
3.02      691
3.03        2
3.09       30
3.10      665
3.11        4
3.20       72
3.30       55
3.40       11

Largest elapsed times:
[3.3999699999999997, 3.3999699999999997, 3.39996, 3.399

In [21]:
# Step 2.5b: compact horizon-quality summary
for col, target in [("elapsed_30", 3.0), ("elapsed_80", 8.0)]:
    s = horizon_check[col].dropna()
    err = (s - target).abs()
    print(f"\n{col}")
    print("total            =", len(s))
    print("mean elapsed     =", round(s.mean(), 6))
    print("median elapsed   =", round(s.median(), 6))
    print("min elapsed      =", round(s.min(), 6))
    print("max elapsed      =", round(s.max(), 6))
    print("within ±0.01 s   =", int((err <= 0.01).sum()))
    print("within ±0.05 s   =", int((err <= 0.05).sum()))
    print("outside ±0.05 s  =", int((err > 0.05).sum()))
    print("outside ±0.10 s  =", int((err > 0.10).sum()))


elapsed_30
total            = 73200
mean elapsed     = 3.001281
median elapsed   = 3.0
min elapsed      = 2.9592
max elapsed      = 3.39997
within ±0.01 s   = 69323
within ±0.05 s   = 72363
outside ±0.05 s  = 837
outside ±0.10 s  = 522

elapsed_80
total            = 13200
mean elapsed     = 8.003413
median elapsed   = 8.0
min elapsed      = 7.95003
max elapsed      = 8.60005
within ±0.01 s   = 11679
within ±0.05 s   = 12894
outside ±0.05 s  = 306
outside ±0.10 s  = 165


In [22]:
# Step 3 (CORE v1.2):
# DuckDB analysis + leakage-safe features + time-validated 3s/8s targets
HORIZON_TOL = 0.05  # seconds
# Safe re-run
try:
    con.close()
except Exception:
    pass
con = duckdb.connect("data/processed/ady.duckdb")
con.register("wm_core_df", df)
SQL_CORE = f"""
CREATE OR REPLACE TABLE wm AS
SELECT * FROM wm_core_df;
-- Q1: basic driving-state summary
SELECT
    AVG((speed < 0.5)::INT) AS stopped,
    AVG((ABS(yaw_rate) > 0.2)::INT) AS turning,
    AVG((acc > 1)::INT) AS accelerating,
    COUNT(*) AS n
FROM wm;
-- Q2: speed by scene-density bin
SELECT
    FLOOR(n_agents / 10) * 10 AS agents_bin,
    AVG(speed) AS speed,
    COUNT(*) AS n
FROM wm
GROUP BY 1
ORDER BY 1;
-- Build leakage-safe feature table.
-- Future x/y/ts are used only transiently for target construction.
CREATE OR REPLACE TABLE feat AS
WITH future_values AS (
    SELECT
        unit,
        scenario,
        step,
        ts,
        x,
        y,
        vx,
        vy,
        heading,
        length,
        width,
        n_agents,
        speed,
        acc,
        yaw_rate,
        -- 10 steps back: approximately 1 s at nominal 10 Hz.
        -- Name explicitly says "10step" because dt is not perfectly uniform.
        speed - LAG(speed, 10) OVER (
            PARTITION BY unit
            ORDER BY step
        ) AS dspeed_10step,
        -- True trailing 1.0-second windows based on timestamp.
        -- Current/past only -> no future leakage.
        AVG(acc) OVER (
            PARTITION BY unit
            ORDER BY ts
            RANGE BETWEEN 1.0 PRECEDING AND CURRENT ROW
        ) AS acc_ma1s,
        AVG(yaw_rate) OVER (
            PARTITION BY unit
            ORDER BY ts
            RANGE BETWEEN 1.0 PRECEDING AND CURRENT ROW
        ) AS yaw_ma1s,
        LEAD(x, 30) OVER w AS x_lead30,
        LEAD(y, 30) OVER w AS y_lead30,
        LEAD(ts, 30) OVER w AS ts_lead30,
        LEAD(x, 80) OVER w AS x_lead80,
        LEAD(y, 80) OVER w AS y_lead80,
        LEAD(ts, 80) OVER w AS ts_lead80
    FROM wm
    WINDOW w AS (
        PARTITION BY unit
        ORDER BY step
    )
)
SELECT
    unit,
    scenario,
    step,
    ts,
    x,
    y,
    vx,
    vy,
    heading,
    length,
    width,
    n_agents,
    speed,
    acc,
    yaw_rate,
    dspeed_10step,
    acc_ma1s,
    yaw_ma1s,
    CASE
        WHEN ts_lead30 IS NOT NULL
         AND ABS((ts_lead30 - ts) - 3.0) <= {HORIZON_TOL}
        THEN SQRT(
            POWER(x_lead30 - x, 2)
            + POWER(y_lead30 - y, 2)
        )
        ELSE NULL
    END AS y_disp_3s,
    CASE
        WHEN ts_lead80 IS NOT NULL
         AND ABS((ts_lead80 - ts) - 8.0) <= {HORIZON_TOL}
        THEN SQRT(
            POWER(x_lead80 - x, 2)
            + POWER(y_lead80 - y, 2)
        )
        ELSE NULL
    END AS y_disp_8s
FROM future_values;
-- Q3: target-horizon quality
SELECT
    COUNT(*) AS n_rows,
    SUM((step <= 60)::INT) AS lead30_possible,
    COUNT(y_disp_3s) AS n3_valid,
    ROUND(
        100.0 * COUNT(y_disp_3s)
        / NULLIF(SUM((step <= 60)::INT), 0),
        2
    ) AS pct3_retained,
    SUM((step <= 10)::INT) AS lead80_possible,
    COUNT(y_disp_8s) AS n8_valid,
    ROUND(
        100.0 * COUNT(y_disp_8s)
        / NULLIF(SUM((step <= 10)::INT), 0),
        2
    ) AS pct8_retained
FROM feat;
"""
# Execute each SQL statement and save SELECT outputs
k = 0
SQL_NO_COMMENTS = "\n".join(line for line in SQL_CORE.splitlines() if not line.strip().startswith("--"))
for stmt in SQL_NO_COMMENTS.split(";"):
    stmt = stmt.strip()
    if not stmt:
        continue
    res = con.execute(stmt)
    if stmt.upper().startswith(("SELECT", "WITH")):
        k += 1
        out = res.df()
        out.to_csv(f"report/table_q{k}.csv", index=False)
        print(f"\n--- Q{k} ---")
        print(out.head(20))
# Save SQL used in the experiment
with open("sql/queries.sql", "w") as f:
    f.write(SQL_CORE)
# Export final core feature table
feat = con.execute("SELECT * FROM feat").df()
feat.to_parquet("data/processed/feat_v1_2.parquet", index=False)
print("\nSaved data/processed/feat_v1_2.parquet", feat.shape)


--- Q1 ---
    stopped   turning  accelerating       n
0  0.686538  0.028824      0.120315  109200

--- Q2 ---
    agents_bin     speed      n
0          0.0  9.203373   2284
1         10.0  6.599121   9334
2         20.0  3.624136  13905
3         30.0  2.646942  16544
4         40.0  2.178553  16877
5         50.0  1.692835  12171
6         60.0  1.545598   8248
7         70.0  1.627682   6107
8         80.0  0.701824   5147
9         90.0  0.907008   4619
10       100.0  0.789077   3643
11       110.0  0.224090   4317
12       120.0  0.254505   1917
13       130.0  0.340308    820
14       140.0  0.055371    952
15       150.0  0.000872    512
16       160.0  0.000260    354
17       170.0  0.000000    315
18       180.0  0.000000    520
19       190.0  0.000000    309

--- Q3 ---
   n_rows  lead30_possible  n3_valid  pct3_retained  lead80_possible  n8_valid  pct8_retained
0  109200          73200.0     72363          98.86          13200.0     12894          97.68

Saved data/proc

### 3.2. RQ1 — scene density, high-speed events và turning intensity

Bản cũ dùng `period = step`, nên không trả lời đúng RQ1. Bản này chỉ dùng core `feat_v1_2.parquet` và tính trực tiếp density/turning theo đúng câu hỏi nghiên cứu.


In [23]:
# Step 3.1 (CORE): audit validated future targets
print("Feature table shape:", feat.shape)
print("Units:", feat["unit"].nunique())
print("\nColumns:")
print(feat.columns.tolist())
# Target availability
n3 = int(feat["y_disp_3s"].notna().sum())
n8 = int(feat["y_disp_8s"].notna().sum())
print("\nValid targets:")
print("3s =", n3)
print("8s =", n8)
assert feat.shape == (EXPECTED_CORE["rows"], 20)
assert feat["unit"].nunique() == EXPECTED_CORE["units"]
assert n3 == EXPECTED_CORE["n3"]
assert n8 == EXPECTED_CORE["n8"]
# No future helper columns may survive
future_helper_cols = [
    c for c in feat.columns if any(k in c for k in ["lead30", "lead80", "ts_lead", "x_lead", "y_lead"])
]
print("\nFuture helper columns:", future_helper_cols)
assert future_helper_cols == []
# Targets must be valid distances
assert (feat["y_disp_3s"].dropna() >= 0).all()
assert (feat["y_disp_8s"].dropna() >= 0).all()
print("\n3s target distribution:")
print(feat["y_disp_3s"].dropna().describe())
print("\n8s target distribution:")
print(feat["y_disp_8s"].dropna().describe())
# Coverage across trajectories
coverage = feat.groupby("unit").agg(
    n_rows=("step", "size"), valid_3s=("y_disp_3s", "count"), valid_8s=("y_disp_8s", "count")
)
print("\nCoverage per trajectory:")
print(coverage.describe())
print("\nUnits with at least one valid target:")
print("3s:", int((coverage["valid_3s"] > 0).sum()), "/", EXPECTED_CORE["units"])
print("8s:", int((coverage["valid_8s"] > 0).sum()), "/", EXPECTED_CORE["units"])
print("\nTarget audit: OK")

Feature table shape: (109200, 20)
Units: 1200

Columns:
['unit', 'scenario', 'step', 'ts', 'x', 'y', 'vx', 'vy', 'heading', 'length', 'width', 'n_agents', 'speed', 'acc', 'yaw_rate', 'dspeed_10step', 'acc_ma1s', 'yaw_ma1s', 'y_disp_3s', 'y_disp_8s']

Valid targets:
3s = 72363
8s = 12894

Future helper columns: []

3s target distribution:
count    72363.000000
mean         7.275476
std         14.036110
min          0.000000
25%          0.000000
50%          0.000000
75%          7.905826
max         84.640893
Name: y_disp_3s, dtype: float64

8s target distribution:
count    12894.000000
mean        19.456142
std         36.473054
min          0.000000
25%          0.000000
50%          0.000000
75%         23.205672
max        223.729558
Name: y_disp_8s, dtype: float64

Coverage per trajectory:
       n_rows     valid_3s     valid_8s
count  1200.0  1200.000000  1200.000000
mean     91.0    60.302500    10.745000
std       0.0     4.982806     1.632645
min      91.0     0.000000     0.

In [24]:
# Step 3.1b (CORE): inspect target coverage after timing validation
coverage = feat.groupby("unit").agg(
    n_rows=("step", "size"), valid_3s=("y_disp_3s", "count"), valid_8s=("y_disp_8s", "count")
)
print("3s valid-target counts per trajectory:")
print(coverage["valid_3s"].describe())
print("\n8s valid-target counts per trajectory:")
print(coverage["valid_8s"].describe())
# Units that lost all targets after timing validation
zero_3s = coverage[coverage["valid_3s"] == 0]
zero_8s = coverage[coverage["valid_8s"] == 0]
print("\nUnits with ZERO valid 3s targets:", len(zero_3s))
print(zero_3s)
print("\nUnits with ZERO valid 8s targets:", len(zero_8s))
print(zero_8s)
# Frequency distribution of valid target counts
print("\nFrequency of valid 3s counts:")
print(coverage["valid_3s"].value_counts().sort_index())
print("\nFrequency of valid 8s counts:")
print(coverage["valid_8s"].value_counts().sort_index())
# Coverage ratios
coverage["coverage_3s"] = coverage["valid_3s"] / 61
coverage["coverage_8s"] = coverage["valid_8s"] / 11
print("\nCoverage-ratio summary:")
print(coverage[["coverage_3s", "coverage_8s"]].describe().round(4))

3s valid-target counts per trajectory:
count    1200.000000
mean       60.302500
std         4.982806
min         0.000000
25%        61.000000
50%        61.000000
75%        61.000000
max        61.000000
Name: valid_3s, dtype: float64

8s valid-target counts per trajectory:
count    1200.000000
mean       10.745000
std         1.632645
min         0.000000
25%        11.000000
50%        11.000000
75%        11.000000
max        11.000000
Name: valid_8s, dtype: float64

Units with ZERO valid 3s targets: 1
                     n_rows  valid_3s  valid_8s
unit                                           
5cb8a511548cd234_31      91         0         0

Units with ZERO valid 8s targets: 25
                       n_rows  valid_3s  valid_8s
unit                                             
1491328b7f06e41d_3580      91        34         0
194c7292ea2aebfd_3189      91         5         0
2a59ed5a3b80f350_1280      91         7         0
2f4ddb476890748f_1226      91        34         0
2fb0

In [25]:
# Step 3.1c (CORE): summary tables for validated 3s / 8s targets
# 1. Overall target summary by horizon
target_summary = pd.DataFrame(
    [
        {
            "horizon": "3s",
            "n_valid": feat["y_disp_3s"].count(),
            "mean": feat["y_disp_3s"].mean(),
            "median": feat["y_disp_3s"].median(),
            "std": feat["y_disp_3s"].std(),
            "p05": feat["y_disp_3s"].quantile(0.05),
            "p25": feat["y_disp_3s"].quantile(0.25),
            "p75": feat["y_disp_3s"].quantile(0.75),
            "p95": feat["y_disp_3s"].quantile(0.95),
            "max": feat["y_disp_3s"].max(),
        },
        {
            "horizon": "8s",
            "n_valid": feat["y_disp_8s"].count(),
            "mean": feat["y_disp_8s"].mean(),
            "median": feat["y_disp_8s"].median(),
            "std": feat["y_disp_8s"].std(),
            "p05": feat["y_disp_8s"].quantile(0.05),
            "p25": feat["y_disp_8s"].quantile(0.25),
            "p75": feat["y_disp_8s"].quantile(0.75),
            "p95": feat["y_disp_8s"].quantile(0.95),
            "max": feat["y_disp_8s"].max(),
        },
    ]
)
target_summary.round(4).to_csv("report/table_target_summary.csv", index=False)
print("Overall target summary:")
print(target_summary.round(4).to_string(index=False))
# 2. Target statistics by trajectory
target_by_unit = (
    feat.groupby("unit")
    .agg(
        n_3s=("y_disp_3s", "count"),
        mean_3s=("y_disp_3s", "mean"),
        median_3s=("y_disp_3s", "median"),
        n_8s=("y_disp_8s", "count"),
        mean_8s=("y_disp_8s", "mean"),
        median_8s=("y_disp_8s", "median"),
    )
    .reset_index()
)
target_by_unit.round(4).to_csv("report/table_target_by_unit.csv", index=False)
print("\nTarget-by-unit shape:", target_by_unit.shape)
print(target_by_unit.head(10).round(4).to_string(index=False))
# 3. Target statistics by current trajectory step
target_by_step = (
    feat.groupby("step")
    .agg(
        n_3s=("y_disp_3s", "count"),
        mean_3s=("y_disp_3s", "mean"),
        median_3s=("y_disp_3s", "median"),
        n_8s=("y_disp_8s", "count"),
        mean_8s=("y_disp_8s", "mean"),
        median_8s=("y_disp_8s", "median"),
    )
    .reset_index()
)
target_by_step.round(4).to_csv("report/table_target_by_step.csv", index=False)
print("\nTarget-by-step first 15 rows:")
print(target_by_step.head(15).round(4).to_string(index=False))
# Integrity checks
assert int(target_summary.loc[target_summary["horizon"] == "3s", "n_valid"].iloc[0]) == EXPECTED_CORE["n3"]
assert int(target_summary.loc[target_summary["horizon"] == "8s", "n_valid"].iloc[0]) == EXPECTED_CORE["n8"]
assert len(target_by_unit) == EXPECTED_CORE["units"]
assert len(target_by_step) == 91
print("\nValidated target summary tables: OK")

Overall target summary:
horizon  n_valid    mean  median     std  p05  p25     p75      p95      max
     3s    72363  7.2755     0.0 14.0361  0.0  0.0  7.9058  38.9380  84.6409
     8s    12894 19.4561     0.0 36.4731  0.0  0.0 23.2057 102.0154 223.7296

Target-by-unit shape: (1200, 7)
                 unit  n_3s  mean_3s  median_3s  n_8s  mean_8s  median_8s
 1038e39306f1e9e2_329    61   0.0000     0.0000    11   0.0000     0.0000
  103cca79b487b34_858    61   0.0000     0.0000    11   0.0000     0.0000
  103cca79b487b34_872    61   0.0000     0.0000    11   0.0000     0.0000
 1067273616585b8f_207    61   0.0000     0.0000    11   0.0000     0.0000
10abcef49bdb019c_4250    61   2.1621     2.2297    11   5.1813     5.1863
 10c766c45d554705_254    61   0.0000     0.0000    11   0.0000     0.0000
111badd033a2be3b_3445    61   0.0000     0.0000    11   0.0000     0.0000
  11201d60b13e12bf_13    61   0.0000     0.0000    11   0.0000     0.0000
  11201d60b13e12bf_26    61   0.0000     0.000

In [26]:
# Step 3.1d (CORE): audit zero-displacement targets
EPS = 1e-6
for target in ["y_disp_3s", "y_disp_8s"]:
    d = feat.dropna(subset=[target]).copy()
    zero = d[target].abs() <= EPS
    nonzero = ~zero
    print("\n" + "=" * 60)
    print(target)
    print("=" * 60)
    print("valid targets      =", len(d))
    print("zero displacement  =", int(zero.sum()))
    print("zero share         =", round(zero.mean(), 4))
    print("\nCurrent speed when target displacement = 0:")
    print(d.loc[zero, "speed"].describe().round(4))
    print("\nCurrent speed when target displacement > 0:")
    print(d.loc[nonzero, "speed"].describe().round(4))
    print(
        "\nZero-target rows with current speed < 0.5 m/s =",
        int((d.loc[zero, "speed"] < 0.5).sum()),
        "/",
        int(zero.sum()),
    )
    print("Zero-target rows with current speed >= 0.5 m/s =", int((d.loc[zero, "speed"] >= 0.5).sum()))
# docx 3.3: save the stationary/zero-displacement facts as a table for the paper
zero_rows = []
for target in ["y_disp_3s", "y_disp_8s"]:
    d0 = feat.dropna(subset=[target])
    zero = d0[target].abs() <= ZERO_DISP_EPS
    zero_rows.append(
        {
            "target": target,
            "valid_rows": len(d0),
            "vehicles": int(d0["unit"].nunique()),
            "zero_displacement_rows": int(zero.sum()),
            "zero_share": float(zero.mean()),
            "zero_rows_with_speed_below_0_5": int((d0.loc[zero, "speed"] < STOP_SPEED).sum()),
            "stationary_share_speed_below_0_5": float((d0["speed"] < STOP_SPEED).mean()),
        }
    )
zero_table = pd.DataFrame(zero_rows)
zero_table.to_csv("report/table_zero_displacement.csv", index=False)
display(zero_table.round(4))


y_disp_3s
valid targets      = 72363
zero displacement  = 40351
zero share         = 0.5576

Current speed when target displacement = 0:
count    40351.0
mean         0.0
std          0.0
min          0.0
25%          0.0
50%          0.0
75%          0.0
max          0.0
Name: speed, dtype: float64

Current speed when target displacement > 0:
count    32012.0000
mean         5.4257
std          5.7778
min          0.0000
25%          0.2620
50%          3.5269
75%          8.9254
max         35.8806
Name: speed, dtype: float64

Zero-target rows with current speed < 0.5 m/s = 40351 / 40351
Zero-target rows with current speed >= 0.5 m/s = 0

y_disp_8s
valid targets      = 12894
zero displacement  = 7181
zero share         = 0.5569

Current speed when target displacement = 0:
count    7181.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
Name: speed, dtype: float64

Current speed when target displacement > 0:
count    5713

,target,valid_rows,vehicles,zero_displacement_rows,zero_share,zero_rows_with_speed_below_0_5,stationary_share_speed_below_0_5
0,y_disp_3s,72363,1199,40351,0.5576,40351,0.6856
1,y_disp_8s,12894,1175,7181,0.5569,7181,0.6868


### 3.3. Bảng dự báo một thời điểm (góp ý 1a, 3.3, 5.5)

**Vì sao:** mục tiêu 8 s chỉ tồn tại ở step 0–10, còn mục tiêu 3 s ở step 0–60. Bản cũ vì vậy so 3 s và 8 s trên hai bộ dòng khác nhau, khác cả số biến (8 và 7) lẫn cách chia tập.

**Sửa:** mỗi xe chỉ giữ **một dòng tại step 10** (hết giây thứ 1, đủ 1 s lịch sử cho `dspeed_10step` và các trung bình trượt). Từ dòng đó dự báo cả 3 s và 8 s với cùng 8 biến. Notebook B/D phải dùng `feat_core_origin10.parquet` và **chia tập một lần** cho cả hai horizon.

**Lưu ý:** bảng này nhỏ (khoảng một dòng một xe), nên ở Notebook D phải báo khoảng tin cậy bootstrap theo cảnh.

In [27]:
# Step 3.3 (CORE, docx 1a + 3.3 + 5.5): one forecast origin per vehicle, shared by 3 s and 8 s
origin_core, origin_core_audit = build_origin_table(feat, label="v1_2 core")
origin_core_audit.to_csv("report/table_core_origin_audit.csv", index=False)
display(origin_core_audit.T)
# Same rows, same predictors for both horizons
assert origin_core["y_disp_3s"].notna().all() and origin_core["y_disp_8s"].notna().all()
assert (origin_core["step"] == ORIGIN_STEP).all()
# docx 3.3: stationary vs moving, with the constant-velocity error inside each group
motion_rows = []
for state, g in origin_core.groupby("motion_state"):
    row = {
        "motion_state": state,
        "vehicles": len(g),
        "share": len(g) / len(origin_core),
        "scenarios": g["scenario"].nunique(),
    }
    for h, col in [(3.0, "y_disp_3s"), (8.0, "y_disp_8s")]:
        tag = f"{int(h)}s"
        row[f"mean_disp_{tag}"] = g[col].mean()
        row[f"median_disp_{tag}"] = g[col].median()
        row[f"zero_disp_share_{tag}"] = (g[col] <= ZERO_DISP_EPS).mean()
        row[f"cv_MAE_{tag}"] = (g[col] - g["speed"] * h).abs().mean()
    motion_rows.append(row)
origin_motion = pd.DataFrame(motion_rows)
origin_motion.to_csv("report/table_core_origin_motion.csv", index=False)
display(origin_motion.round(4))
# docx 5.5: report vehicles and scenarios next to rows
veh_per_scen_origin = origin_core.groupby("scenario")["unit"].nunique()
print(
    "Origin table rows:",
    len(origin_core),
    "| vehicles:",
    origin_core["unit"].nunique(),
    "| scenarios:",
    origin_core["scenario"].nunique(),
    "| vehicles per scenario (mean/max):",
    round(veh_per_scen_origin.mean(), 3),
    "/",
    veh_per_scen_origin.max(),
)
origin_core.to_parquet("data/processed/feat_core_origin10.parquet", index=False)
print("Saved data/processed/feat_core_origin10.parquet", origin_core.shape)

,0
table,v1_2 core
origin_step,10
origin_ts_median_s,1.0
vehicles_at_origin,1200
missing_3s_target,15
missing_8s_target,28
missing_predictor,0
final_rows,1172
final_vehicles,1172
final_scenarios,1051


,motion_state,vehicles,share,scenarios,mean_disp_3s,median_disp_3s,zero_disp_share_3s,cv_MAE_3s,mean_disp_8s,median_disp_8s,zero_disp_share_8s,cv_MAE_8s
0,moving,364,0.3106,355,22.4502,18.5674,0.0000,3.9188,58.2099,52.8087,0.0000,22.0692
1,stationary,808,0.6894,712,0.1989,0.0000,0.8082,0.1953,2.2022,0.0000,0.8082,2.1291


Origin table rows: 1172 | vehicles: 1172 | scenarios: 1051 | vehicles per scenario (mean/max): 1.115 / 4
Saved data/processed/feat_core_origin10.parquet (1172, 23)


In [28]:
# Step 3.2 (CORE): RQ1 — scene density, high-speed events and turning intensity
from scipy.stats import kruskal
q1 = pd.read_parquet("data/processed/feat_v1_2.parquet")
d = q1.dropna(subset=["speed", "n_agents", "yaw_rate"]).copy()
print("RQ1 usable rows:", len(d))
print("RQ1 units:", d["unit"].nunique())
# A. Scene density groups
q33, q66 = d["n_agents"].quantile([1 / 3, 2 / 3])
d["density_group"] = np.select(
    [d["n_agents"] <= q33, d["n_agents"] <= q66], ["Low", "Medium"], default="High"
)
d["density_group"] = pd.Categorical(d["density_group"], categories=["Low", "Medium", "High"], ordered=True)
print("\nDensity cut points:")
print("q33 =", round(q33, 3))
print("q66 =", round(q66, 3))
# B. High-speed events
speed_threshold = d["speed"].quantile(0.90)
d["high_speed_event"] = d["speed"] > speed_threshold
density_table = (
    d.groupby("density_group", observed=True)
    .agg(
        event_share=("high_speed_event", "mean"),
        mean_speed=("speed", "mean"),
        median_speed=("speed", "median"),
        observations=("speed", "size"),
        units=("unit", "nunique"),
    )
    .reset_index()
)
corr_density = d["speed"].corr(d["n_agents"])
# C. Kruskal test at TRAJECTORY level
#    Avoid treating every timestep as independent.
unit_density = (
    d.groupby("unit").agg(mean_speed=("speed", "mean"), mean_n_agents=("n_agents", "mean")).reset_index()
)
uq33, uq66 = unit_density["mean_n_agents"].quantile([1 / 3, 2 / 3])
unit_density["density_group"] = np.select(
    [unit_density["mean_n_agents"] <= uq33, unit_density["mean_n_agents"] <= uq66],
    ["Low", "Medium"],
    default="High",
)
density_groups = [g["mean_speed"].values for _, g in unit_density.groupby("density_group")]
kw_density = kruskal(*density_groups)
# D. Turning intensity
#    0 = straight / essentially no turning
#    1-10 = deciles among rows with positive turning
TURN_EPS = 1e-6
d["abs_yaw_rate"] = d["yaw_rate"].abs()
straight_mask = d["abs_yaw_rate"] <= TURN_EPS
turn_mask = ~straight_mask
d["turning_decile"] = pd.Series(pd.NA, index=d.index, dtype="Int64")
# Group 0 = no measurable turning
d.loc[straight_mask, "turning_decile"] = 0
# Deciles 1-10 only among genuinely turning observations
positive_turn = d.loc[turn_mask, "abs_yaw_rate"]
positive_decile = pd.qcut(positive_turn, q=10, labels=False, duplicates="drop") + 1
d.loc[turn_mask, "turning_decile"] = positive_decile.astype("Int64")
turning_table = (
    d.groupby("turning_decile", observed=True)
    .agg(
        mean_speed=("speed", "mean"),
        median_speed=("speed", "median"),
        mean_abs_yaw_rate=("abs_yaw_rate", "mean"),
        observations=("speed", "size"),
        units=("unit", "nunique"),
    )
    .reset_index()
)
corr_turning = d["speed"].corr(d["abs_yaw_rate"])
print("\nStraight rows:", int(straight_mask.sum()))
print("Positive-turn rows:", int(turn_mask.sum()))
print("Occupied turning groups:", turning_table["turning_decile"].tolist())
# Save report tables
density_table.round(4).to_csv("report/table_rq1_density.csv", index=False)
turning_table.round(4).to_csv("report/table_rq1_turning.csv", index=False)
unit_density.round(4).to_csv("report/table_rq1_density_unit.csv", index=False)
# Output
print("\n90th-percentile speed threshold:", round(speed_threshold, 4), "m/s")
print("Pearson corr(speed, density):", round(corr_density, 4))
print("Kruskal trajectory-level density groups:", kw_density)
print("Pearson corr(speed, |yaw_rate|):", round(corr_turning, 4))
print("Occupied turning deciles:", turning_table["turning_decile"].tolist())
print("\nDensity table:")
display(density_table.round(4))
print("\nTurning table:")
display(turning_table.round(4))
assert len(d) == EXPECTED_CORE["rows"] - EXPECTED_CORE["units"]
assert d["unit"].nunique() == EXPECTED_CORE["units"]
assert len(density_table) == 3
positive_bins = turning_table.loc[turning_table["turning_decile"] > 0, "turning_decile"].nunique()
assert 0 in turning_table["turning_decile"].values
assert positive_bins == 10
print("\nRQ1 Step 3.2: OK")
# Spearman (rank) correlations as well, so the paper states which coefficient it uses
from scipy.stats import spearmanr
rho_density, p_rho_density = spearmanr(d["speed"], d["n_agents"])
rho_turning, p_rho_turning = spearmanr(d["speed"], d["abs_yaw_rate"])
rq1_stats = pd.DataFrame(
    [
        ["rows", len(d)],
        ["units", d["unit"].nunique()],
        ["density_cut_q33", q33],
        ["density_cut_q66", q66],
        ["speed_threshold_p90", speed_threshold],
        ["pearson_speed_density", corr_density],
        ["spearman_speed_density", rho_density],
        ["pearson_speed_abs_yaw", corr_turning],
        ["spearman_speed_abs_yaw", rho_turning],
        ["kruskal_density_unit_level_H", kw_density.statistic],
        ["kruskal_density_unit_level_p", kw_density.pvalue],
        ["straight_rows_group0", int(straight_mask.sum())],
        ["turning_rows_groups1_10", int(turn_mask.sum())],
    ],
    columns=["statistic", "value"],
)
rq1_stats.to_csv("report/table_rq1_stats.csv", index=False)
print("\nSpearman rho(speed, density):", round(rho_density, 4))
print("Spearman rho(speed, |yaw rate|):", round(rho_turning, 4))

RQ1 usable rows: 108000
RQ1 units: 1200

Density cut points:
q33 = 36.0
q66 = 61.0

Straight rows: 60951
Positive-turn rows: 47049
Occupied turning groups: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

90th-percentile speed threshold: 9.8496 m/s
Pearson corr(speed, density): -0.307
Kruskal trajectory-level density groups: KruskalResult(statistic=169.3975408847055, pvalue=1.6435819868998466e-37)
Pearson corr(speed, |yaw_rate|): 0.1663
Occupied turning deciles: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

Density table:


,density_group,event_share,mean_speed,median_speed,observations,units
0,Low,0.1953,4.4661,0.3882,36308,528
1,Medium,0.0729,2.0019,0.0000,36044,568
2,High,0.0304,0.8992,0.0000,35648,486



Turning table:


,turning_decile,mean_speed,median_speed,mean_abs_yaw_rate,observations,units
0,0,0.0012,0.0000,0.0000,60951,704
1,1,4.1619,0.4872,0.0006,4705,485
2,2,6.0067,4.4473,0.0028,4705,490
3,3,6.3283,4.7888,0.0054,4705,485
4,4,6.0633,4.0979,0.0085,4705,489
5,5,6.1734,4.6954,0.0123,4705,490
6,6,6.1903,4.3544,0.0172,4704,487
7,7,6.4420,4.9624,0.0245,4705,485
8,8,6.0161,4.2798,0.0376,4705,459
9,9,4.5754,3.0577,0.0748,4705,382



RQ1 Step 3.2: OK

Spearman rho(speed, density): -0.4069
Spearman rho(speed, |yaw rate|): 0.9027


## Bước 5 (phần A): hình RQ1 (thay Fig. 2 của bài)

Hình vẽ từ dữ liệu dùng để huấn luyện mô hình. Theo file góp ý (4.3, 4.7) và tài liệu trực quan hoá:
- Vẽ đúng kích thước in (rộng 12.2 cm), chữ 8 pt; nền trắng; không có tiêu đề trong hình, mỗi khung chỉ có nhãn (a), (b).
- (a) có một thông điệp, nên chỉ cột "Low" mang màu nhấn san hô; số trên cột ghi phần trăm với một chữ số thập phân; trục bắt đầu từ 0.
- (b) nhóm 0 (không quay đầu, gồm cả xe đứng yên) được tách riêng, không nối đường với nhóm 1-10; vẽ cả trung bình và trung vị; nhãn đặt cạnh dữ liệu, không dùng khung chú giải; phân biệt bằng hình dạng điểm và kiểu nét, không chỉ bằng màu.
- Caption mở đầu bằng thông điệp có hướng và độ lớn, rồi giải thích từng khung và điều kiện đo.
- Nếu bài thiếu chỗ (mục 7 của file góp ý), dùng đoạn văn trong `rq1_text_option.txt` thay cho hình.

In [29]:
# Step 5A (CORE): RQ1 figure following docx 4.3/4.7 and the visualisation guide (Sec. 1.2, 2.2-2.5, 2.9, 2.13)
order = ["Low", "Medium", "High"]
dtab = density_table.assign(g=density_table["density_group"].astype(str)).set_index("g").loc[order]
share_pct = 100 * dtab["event_share"]
tt = turning_table.assign(g=turning_table["turning_decile"].astype(int)).sort_values("g")
g0, g1 = tt[tt["g"] == 0].iloc[0], tt[tt["g"] > 0]
ratio = dtab.loc["Low", "event_share"] / dtab.loc["High", "event_share"]
fig, axes = plt.subplots(1, 2, figsize=(FIG_W, 2.2), gridspec_kw={"width_ratios": [1, 1.7]})
# (a) one claim: high-speed rows are far more common in sparse scenes -> accent only on "Low"; bars start at 0
ax = axes[0]
bars = ax.bar(order, share_pct, width=0.6, color=[C_ACCENT, C_BASE, C_BASE], edgecolor=C_TEXT, linewidth=0.4)
ax.bar_label(bars, labels=[f"{v:.1f}%" for v in share_pct], padding=1.5)
ax.set_ylim(0, share_pct.max() * 1.22)
ax.set_xlabel("Scene density")
ax.set_ylabel("High-speed rows (%)")
ax.text(-0.32, 1.02, "(a)", transform=ax.transAxes)
# (b) group 0 is a separate category (not a decile): drawn apart, not joined by the line;
#     mean AND median shown because the mean hides the skewed distribution; labels placed next to the data
ax = axes[1]
x0 = -0.8
ax.plot(g1["g"], g1["mean_speed"], color=C_TEXT, marker="o", ms=3, lw=1)
ax.plot(g1["g"], g1["median_speed"], color=C_MID, marker="^", ms=3, lw=1, ls="--")
ax.plot([x0], [g0["mean_speed"]], color=C_TEXT, marker="s", ms=4, ls="none")
ax.plot([x0], [g0["median_speed"]], color=C_MID, marker="s", ms=4, ls="none", mec=C_TEXT, mew=0.4)
ax.axvline(0.1, color=C_TEXT, lw=0.4, ls=":")
mid = g1[(g1["g"] >= 3) & (g1["g"] <= 8)]  # label inside the line, where the two lines are furthest apart
k = int(g1.index.get_loc((mid["mean_speed"] - mid["median_speed"]).idxmax()))
ax.text(g1["g"].iloc[k], g1["mean_speed"].iloc[k] + 0.35, "mean", ha="center", va="bottom")
ax.text(g1["g"].iloc[k], g1["median_speed"].iloc[k] - 0.35, "median", ha="center", va="top", color=C_MID)
ax.annotate("no turning\n(incl. stopped)", xy=(x0, g0["mean_speed"]), xytext=(x0 + 0.5, max(tt["mean_speed"]) * 0.25),
            fontsize=7, arrowprops={"arrowstyle": "-", "lw": 0.4, "color": C_TEXT})
ax.set_xticks([x0] + list(range(1, 11)))
ax.set_xticklabels(["0"] + [str(k) for k in range(1, 11)])
ax.set_xlim(x0 - 0.6, 10.6)
ax.set_ylim(0, max(tt["mean_speed"].max(), tt["median_speed"].max()) * 1.15)
ax.set_xlabel("Turning group (0 = none; 1-10 = |yaw rate| decile)")
ax.set_ylabel("Speed (m/s)")
ax.text(-0.2, 1.02, "(b)", transform=ax.transAxes)
fig.tight_layout(pad=0.4, w_pad=1.0)
fig.savefig("report/fig_rq1_density_turning.png", bbox_inches="tight")
fig.savefig("report/fig_rq1_density_turning.pdf", bbox_inches="tight")
plt.show()
# Caption (guide 2.4): message first (direction + size), then each panel, then measurement conditions
caption = (f"Fig. 2. High-speed rows are {ratio:.1f} times more frequent in low-density than in high-density scenes "
    f"({share_pct['Low']:.1f}% vs {share_pct['High']:.1f}%), while among turning rows mean speed shows no monotonic trend with turning "
    f"intensity (between {g1['mean_speed'].min():.1f} and {g1['mean_speed'].max():.1f} m/s). "
    f"(a) Share of rows faster than the 90th-percentile speed ({speed_threshold:.1f} m/s) by scene-density tercile "
    f"(at most {q33:.0f}, {q33:.0f}-{q66:.0f} and more than {q66:.0f} agents). "
    f"(b) Mean (solid) and median (dashed) speed by turning group; group 0 holds rows with no heading change, "
    f"including stopped vehicles, and groups 1-10 are deciles of |yaw rate| among the other rows. "
    f"Data: {d['unit'].nunique():,} WOMD v1.2 vehicle trajectories, {len(d):,} rows at 10 Hz.")
open("report/fig_rq1_caption.txt", "w", encoding="utf-8").write(caption)
# Option without a figure (guide 1.3: two or three numbers fit in a sentence)
text_option = (f"High-speed rows (above {speed_threshold:.1f} m/s) make up {share_pct['Low']:.1f}% of rows in low-density "
    f"scenes, {share_pct['Medium']:.1f}% in medium-density scenes and {share_pct['High']:.1f}% in high-density scenes "
    f"(Spearman rho between speed and scene density {rho_density:.3f}; trajectory-level Kruskal-Wallis p = {kw_density.pvalue:.1e}). "
    f"Among turning rows, mean speed stays between {g1['mean_speed'].min():.1f} and {g1['mean_speed'].max():.1f} m/s across "
    f"|yaw rate| deciles, whereas rows with no heading change average {g0['mean_speed']:.1f} m/s because they include stopped vehicles.")
open("report/rq1_text_option.txt", "w", encoding="utf-8").write(text_option)
print("Caption:\n" + caption)
print("\nText option (if Fig. 2 is dropped):\n" + text_option)
print("\nPearson/Spearman density:", round(corr_density, 4), round(rho_density, 4),
      "| Pearson/Spearman turning:", round(corr_turning, 4), round(rho_turning, 4))
print("Saved: report/fig_rq1_density_turning.png/.pdf, fig_rq1_caption.txt, rq1_text_option.txt")

Caption:
Fig. 2. High-speed rows are 6.4 times more frequent in low-density than in high-density scenes (19.5% vs 3.0%), while among turning rows mean speed shows no monotonic trend with turning intensity (between 4.2 and 6.4 m/s). (a) Share of rows faster than the 90th-percentile speed (9.8 m/s) by scene-density tercile (at most 36, 36-61 and more than 61 agents). (b) Mean (solid) and median (dashed) speed by turning group; group 0 holds rows with no heading change, including stopped vehicles, and groups 1-10 are deciles of |yaw rate| among the other rows. Data: 1,200 WOMD v1.2 vehicle trajectories, 108,000 rows at 10 Hz.

Text option (if Fig. 2 is dropped):
High-speed rows (above 9.8 m/s) make up 19.5% of rows in low-density scenes, 7.3% in medium-density scenes and 3.0% in high-density scenes (Spearman rho between speed and scene density -0.407; trajectory-level Kruskal-Wallis p = 1.6e-37). Among turning rows, mean speed stays between 4.2 and 6.4 m/s across |yaw rate| deciles, where


## Test case và bàn giao của Notebook_A

Bàn giao **hai** bảng cho Notebook B/D:

- `feat_core_origin10.parquet`: **bảng chính để huấn luyện và so 3 s với 8 s** (một dòng mỗi xe tại step 10, góp ý 1a).
- `feat.parquet`: bảng đầy đủ mọi step, chỉ dùng cho chẩn đoán phụ.

Manifest chứa cả khoá `units` và `trajectories` để Notebook B và D cùng đọc được. **Chỉ Notebook A được ghi hai file này.**


In [30]:
# CORE validation and handoff to Notebook B/D
import hashlib, json, os
def md5_of(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()
feat_core = pd.read_parquet("data/processed/feat_v1_2.parquet")
origin_core = pd.read_parquet("data/processed/feat_core_origin10.parquet")
# 1. Full table: structure and validated targets
assert feat_core.shape == (EXPECTED_CORE["rows"], 20)
assert feat_core["unit"].nunique() == EXPECTED_CORE["units"]
assert not feat_core.duplicated(["unit", "step"]).any()
assert feat_core.groupby("unit")["step"].count().eq(91).all()
assert feat_core["step"].min() == 0 and feat_core["step"].max() == 90
n3 = int(feat_core["y_disp_3s"].notna().sum())
n8 = int(feat_core["y_disp_8s"].notna().sum())
assert n3 == EXPECTED_CORE["n3"] and n8 == EXPECTED_CORE["n8"]
assert feat_core["y_disp_3s"].dropna().ge(0).all() and feat_core["y_disp_8s"].dropna().ge(0).all()
coverage = feat_core.groupby("unit").agg(valid_3s=("y_disp_3s", "count"), valid_8s=("y_disp_8s", "count"))
units_with_3s = int((coverage["valid_3s"] > 0).sum())
units_with_8s = int((coverage["valid_8s"] > 0).sum())
assert units_with_3s == EXPECTED_CORE["units3"] and units_with_8s == EXPECTED_CORE["units8"]
future_tokens = ["lead30", "lead80", "ts_lead", "x_lead", "y_lead"]
for t in (feat_core, origin_core):
    assert not [c for c in t.columns if any(k in c for k in future_tokens)], "future helper column leaked"
expected_cols = [
    "unit",
    "scenario",
    "step",
    "ts",
    "x",
    "y",
    "vx",
    "vy",
    "heading",
    "length",
    "width",
    "n_agents",
    "speed",
    "acc",
    "yaw_rate",
    "dspeed_10step",
    "acc_ma1s",
    "yaw_ma1s",
    "y_disp_3s",
    "y_disp_8s",
]
assert feat_core.columns.tolist() == expected_cols
# 2. Origin table (docx 1a): one row per vehicle, both targets, same predictors
assert (origin_core["step"] == ORIGIN_STEP).all()
assert not origin_core.duplicated("unit").any()
assert origin_core[COMMON_X + ["y_disp_3s", "y_disp_8s"]].notna().all().all()
assert set(origin_core["unit"]) <= set(feat_core["unit"])
assert origin_core.columns.tolist() == expected_cols + ["motion_state", "zero_disp_3s", "zero_disp_8s"]
# 3. Handoff files (only Notebook A writes these)
feat_core.to_parquet("data/processed/feat.parquet", index=False)
md5_full = md5_of("data/processed/feat.parquet")
md5_origin = md5_of("data/processed/feat_core_origin10.parquet")
n_scen_full = int(feat_core["scenario"].nunique())
manifest = {
    "dataset_version": "WOMD v1.2",
    "author": "Student A",
    # ---- full table (all steps) ----
    "rows": int(len(feat_core)),
    "columns": feat_core.columns.tolist(),
    "units": int(feat_core["unit"].nunique()),  # key used by Notebook D
    "trajectories": int(feat_core["unit"].nunique()),  # key used by Notebook B
    "scenarios": n_scen_full,
    "shards": int(N_SHARDS),
    "steps_per_trajectory": 91,
    "step_min": int(feat_core["step"].min()),
    "step_max": int(feat_core["step"].max()),
    "timestamp_min": float(feat_core["ts"].min()),
    "timestamp_max": float(feat_core["ts"].max()),
    "unit_col": "unit",
    "time_col": "ts",
    "horizon_steps": [30, 80],
    "horizon_seconds": [3.0, 8.0],
    "horizon_tolerance_seconds": 0.05,
    "target_3s_valid": n3,
    "target_8s_valid": n8,
    "target_3s_units": units_with_3s,
    "target_8s_units": units_with_8s,
    "target_3s_scenarios": int(feat_core.loc[feat_core["y_disp_3s"].notna(), "scenario"].nunique()),
    "target_8s_scenarios": int(feat_core.loc[feat_core["y_disp_8s"].notna(), "scenario"].nunique()),
    "target_definition": (
        "Euclidean future displacement; LEAD 30/80 retained only when actual elapsed time "
        "is within ±0.05 s of 3.0/8.0 seconds"
    ),
    "md5": md5_full,
    # ---- main modelling table (docx 1a) ----
    "model_table": "data/processed/feat_core_origin10.parquet",
    "origin_step": ORIGIN_STEP,
    "origin_time_s_nominal": ORIGIN_STEP * 0.1,
    "origin_feature_cols": COMMON_X,
    "origin_rows": int(len(origin_core)),
    "origin_units": int(origin_core["unit"].nunique()),
    "origin_scenarios": int(origin_core["scenario"].nunique()),
    "origin_stationary_share": float((origin_core["motion_state"] == "stationary").mean()),
    "stationary_speed_threshold": STOP_SPEED,
    "origin_md5": md5_origin,
    # ---- preprocessing settings ----
    "clip_kinematics": CLIP_KINEMATICS,
    "acc_limit": ACC_LIMIT,
    "yaw_rate_limit": YAW_RATE_LIMIT,
    "clip_log": CLIP_LOG.get("v1_2", {}),
    "sampling_rule": (
        "reservoir sampling of 60 complete 91-step vehicle tracks per shard, "
        "20 shards, seed 42 + shard index"
    ),
}
with open("data/processed/manifest.json", "w") as f:
    json.dump(manifest, f, indent=2, default=float)
print("Tests A CORE: OK")
print("Saved: data/processed/feat.parquet (all steps)")
print("Saved: data/processed/feat_core_origin10.parquet (main modelling table)")
print("Saved: data/processed/manifest.json")
print(
    json.dumps(
        {
            k: manifest[k]
            for k in [
                "rows",
                "units",
                "scenarios",
                "target_3s_valid",
                "target_8s_valid",
                "origin_rows",
                "origin_units",
                "origin_scenarios",
                "origin_stationary_share",
            ]
        },
        indent=2,
    )
)

Tests A CORE: OK
Saved: data/processed/feat.parquet (all steps)
Saved: data/processed/feat_core_origin10.parquet (main modelling table)
Saved: data/processed/manifest.json
{
  "rows": 109200,
  "units": 1200,
  "scenarios": 1073,
  "target_3s_valid": 72363,
  "target_8s_valid": 12894,
  "origin_rows": 1172,
  "origin_units": 1172,
  "origin_scenarios": 1051,
  "origin_stationary_share": 0.689419795221843
}


## EXTENSION 1 — WOMD v1.3.0 + traffic-light context

Không ghi đè `feat.parquet` của core; mọi output có hậu tố `_v1_3_0`.


**Lưu ý cho bài:** mẫu v1.3.0 là **một mẫu 1,200 xe khác** với mẫu v1.2 của CORE (quy tắc chọn giống hệt, nhưng khác bản dữ liệu). Vì vậy MAE gốc ở mục 5.4 không bằng MAE của CORE. Bài phải ghi điều này.


In [31]:
files_130, df_raw_130 = load_waymo_vehicle_sample("v1_3_0")
df_130 = preprocess_vehicle_table(df_raw_130, label="v1_3_0")
df_130.to_parquet("data/processed/womd_v1_3_0.parquet", index=False)
print("Saved v1.3.0 base:", df_130.shape)

2026-10-11 20:18:56.200262: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [1]
	 [[{{node Placeholder/_0}}]]


v1_3_0 shard 01/20: selected 60 trajectories


2026-10-11 20:19:00.667954: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [1]
	 [[{{node Placeholder/_0}}]]


v1_3_0 shard 02/20: selected 60 trajectories


2026-10-11 20:19:05.904427: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [1]
	 [[{{node Placeholder/_0}}]]


...

v1_3_0 shard 18/20: selected 60 trajectories


2026-10-11 20:20:19.630226: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [1]
	 [[{{node Placeholder/_0}}]]


v1_3_0 shard 19/20: selected 60 trajectories


2026-10-11 20:20:24.107114: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [1]
	 [[{{node Placeholder/_0}}]]


v1_3_0 shard 20/20: selected 60 trajectories
v1_3_0: 1200 trajectories, 109200 agent-step rows
Saved v1.3.0 base: (109200, 17)


In [32]:
# EXTENSION 1A validation: WOMD v1.3.0 base
print("Shape:", df_130.shape)
print("Trajectories:", df_130["unit"].nunique())
print("Shards:", len(files_130))
counts_130 = df_130.groupby("unit")["step"].size()
print("\nSteps per trajectory:")
print(counts_130.describe())
print("\nDuplicate unit-step:", int(df_130.duplicated(["unit", "step"]).sum()))
print("Non-null dt:", int(df_130["dt"].notna().sum()))
print("Non-positive dt:", int((df_130["dt"].notna() & (df_130["dt"] <= 0)).sum()))
print("\ndt statistics:")
print(df_130["dt"].dropna().describe())
print(
    "\ndt outside 0.095–0.105 s:",
    int(((df_130["dt"].dropna() < 0.095) | (df_130["dt"].dropna() > 0.105)).sum()),
)
assert df_130.shape == (109200, 17)
assert df_130["unit"].nunique() == 1200
assert len(files_130) == 20
assert (counts_130 == 91).all()
assert not df_130.duplicated(["unit", "step"]).any()
assert df_130["dt"].notna().sum() == 108000
assert not (df_130["dt"].notna() & (df_130["dt"] <= 0)).any()
print("\nEXTENSION 1A validation: OK")

Shape: (109200, 17)
Trajectories: 1200
Shards: 20

Steps per trajectory:
count    1200.0
mean       91.0
std         0.0
min        91.0
25%        91.0
50%        91.0
75%        91.0
max        91.0
Name: step, dtype: float64

Duplicate unit-step: 0
Non-null dt: 108000
Non-positive dt: 0

dt statistics:
count    108000.000000
mean          0.100013
std           0.001534
min           0.058660
25%           0.099980
50%           0.099990
75%           0.100020
max           0.200910
Name: dt, dtype: float64

dt outside 0.095–0.105 s: 32

EXTENSION 1A validation: OK


In [33]:
traffic_130 = build_traffic_context(files_130, df_130["scenario"].unique())
df_130_traffic = merge_traffic_context(df_130, traffic_130)
df_130_traffic.to_parquet("data/processed/womd_v1_3_0_traffic.parquet", index=False)
print("Traffic rows:", traffic_130.shape)
print("Saved v1.3.0 + traffic:", df_130_traffic.shape)

2026-10-11 20:20:29.186587: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [1]
	 [[{{node Placeholder/_0}}]]
2026-10-11 20:20:33.502898: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [1]
	 [[{{node Placeholder/_0}}]]
2026-10-11 20:20:38.134732: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [1]
	 [[{{n

Traffic rows: (99827, 7)
Saved v1.3.0 + traffic: (109200, 21)


In [34]:
# EXTENSION 1B validation: traffic context merge
n_scenarios_130 = df_130["scenario"].nunique()
print("Selected scenarios:", n_scenarios_130)
print("Traffic shape:", traffic_130.shape)
print("Merged shape:", df_130_traffic.shape)
print("\nTraffic scenarios:", traffic_130["scenario"].nunique())
print("Duplicate traffic scenario-step:", int(traffic_130.duplicated(["scenario", "step"]).sum()))
traffic_cols = ["n_traffic_lights", "n_stop_lights", "n_caution_lights", "n_go_lights"]
print("\nMissing traffic values:")
print(df_130_traffic[traffic_cols].isna().sum())
print("\nTraffic-light summary:")
print(df_130_traffic[traffic_cols].describe().round(3))
# Integrity checks
assert n_scenarios_130 == 1097
assert traffic_130.shape == (1097 * 91, 7)
assert traffic_130["scenario"].nunique() == 1097
assert not traffic_130.duplicated(["scenario", "step"]).any()
assert df_130_traffic.shape == (109200, 21)
assert len(df_130_traffic) == len(df_130)
assert not df_130_traffic[traffic_cols].isna().any().any()
assert (df_130_traffic[traffic_cols] >= 0).all().all()
print("\nEXTENSION 1B traffic validation: OK")

Selected scenarios: 1097
Traffic shape: (99827, 7)
Merged shape: (109200, 21)

Traffic scenarios: 1097
Duplicate traffic scenario-step: 0

Missing traffic values:
n_traffic_lights    0
n_stop_lights       0
n_caution_lights    0
n_go_lights         0
dtype: int64

Traffic-light summary:
       n_traffic_lights  n_stop_lights  n_caution_lights  n_go_lights
count        109200.000     109200.000        109200.000   109200.000
mean              6.658          3.456             0.202        1.480
std               6.597          4.128             0.973        2.413
min               0.000          0.000             0.000        0.000
25%               0.000          0.000             0.000        0.000
50%               6.000          2.000             0.000        0.000
75%              12.000          6.000             0.000        3.000
max              31.000         26.000            11.000       13.000

EXTENSION 1B traffic validation: OK


In [35]:
# EXTENSION 1C: build validated features for v1.3.0
feat_130 = build_feature_table(df_130, include_traffic=False)
feat_130_traffic = build_feature_table(df_130_traffic, include_traffic=True)
feat_130.to_parquet("data/processed/feat_v1_3_0.parquet", index=False)
feat_130_traffic.to_parquet("data/processed/feat_v1_3_0_traffic.parquet", index=False)
print("feat_v1_3_0:", feat_130.shape)
print("feat_v1_3_0_traffic:", feat_130_traffic.shape)
print("\nValid targets — base:")
print("3s:", feat_130["y_disp_3s"].notna().sum())
print("8s:", feat_130["y_disp_8s"].notna().sum())
print("\nValid targets — traffic:")
print("3s:", feat_130_traffic["y_disp_3s"].notna().sum())
print("8s:", feat_130_traffic["y_disp_8s"].notna().sum())

feat_v1_3_0: (109200, 20)
feat_v1_3_0_traffic: (109200, 24)

Valid targets — base:
3s: 72835
8s: 13031

Valid targets — traffic:
3s: 72835
8s: 13031


### EXTENSION 1D: bảng đèn tín hiệu tại một thời điểm dự báo (góp ý 1a, 1b)

Giống CORE: mỗi xe một dòng tại step 10, đủ cả hai mục tiêu và 8 biến chung, nên **base và base + traffic dùng cùng dòng ở cả 3 s và 8 s**. Notebook B (RQ4) phải dùng hai file `*_origin10.parquet` này và chia tập một lần cho cả hai horizon.

In [36]:
# EXTENSION 1D (docx 1a/1b): single-origin tables for the traffic-signal comparison
origin_130_base, audit_130_base = build_origin_table(feat_130, label="v1_3_0 base")
origin_130_traffic, audit_130_traffic = build_origin_table(
    feat_130_traffic, label="v1_3_0 traffic", extra_cols=TRAFFIC_COLS
)
key_cols = ["scenario", "unit", "step"]
assert origin_130_base[key_cols].equals(
    origin_130_traffic[key_cols]
), "base and traffic must use identical rows"
for c in COMMON_X + ["y_disp_3s", "y_disp_8s"]:
    assert np.allclose(origin_130_base[c], origin_130_traffic[c], equal_nan=True), c
origin_130_base.to_parquet("data/processed/feat_v1_3_0_origin10.parquet", index=False)
origin_130_traffic.to_parquet("data/processed/feat_v1_3_0_traffic_origin10.parquet", index=False)
audit_130 = pd.concat([audit_130_base, audit_130_traffic], ignore_index=True)
audit_130.to_csv("report/table_v1_3_0_origin_audit.csv", index=False)
display(audit_130.T)
print(
    "Rows with any traffic light at origin:",
    int((origin_130_traffic["n_traffic_lights"] > 0).sum()),
    "/",
    len(origin_130_traffic),
)

,0,1
table,v1_3_0 base,v1_3_0 traffic
origin_step,10,10
origin_ts_median_s,1.0,1.0
vehicles_at_origin,1200,1200
missing_3s_target,5,5
missing_8s_target,13,13
missing_predictor,0,0
final_rows,1187,1187
final_vehicles,1187,1187
final_scenarios,1085,1085


Rows with any traffic light at origin: 784 / 1187


## EXTENSION 2 — WOMD v1.3.1 + traffic-light context

Dùng biến riêng `files_131`, `df_131`, `feat_131`; không tái sử dụng biến của v1.3.0.


**Lưu ý:** bảng đèn tín hiệu của v1.3.1 **không được dùng trong bài** (RQ4 dùng v1.3.0). Giữ phần này chỉ để kiểm tra tái lập giữa hai phiên bản. Nếu cần tiết kiệm thời gian chạy thì có thể bỏ, nhưng phải bỏ luôn các kiểm tra v1.3.1 ở ô FINAL VALIDATION.


In [37]:
# EXTENSION 2A: load and preprocess WOMD v1.3.1
files_131, df_raw_131 = load_waymo_vehicle_sample("v1_3_1")
df_131 = preprocess_vehicle_table(df_raw_131, label="v1_3_1")
df_131.to_parquet("data/processed/womd_v1_3_1.parquet", index=False)
print("\nSaved v1.3.1 base:", df_131.shape)
print("Trajectories:", df_131["unit"].nunique())
print("Shards:", len(files_131))
print("\nTime-step statistics:")
print(df_131["dt"].dropna().describe())

2026-10-11 20:21:57.147287: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [1]
	 [[{{node Placeholder/_0}}]]


v1_3_1 shard 01/20: selected 60 trajectories


2026-10-11 20:22:01.659656: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [1]
	 [[{{node Placeholder/_0}}]]


v1_3_1 shard 02/20: selected 60 trajectories


2026-10-11 20:22:06.225541: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [1]
	 [[{{node Placeholder/_0}}]]


...

v1_3_1 shard 18/20: selected 60 trajectories


2026-10-11 20:23:18.652567: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [1]
	 [[{{node Placeholder/_0}}]]


v1_3_1 shard 19/20: selected 60 trajectories


2026-10-11 20:23:22.910299: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [1]
	 [[{{node Placeholder/_0}}]]


v1_3_1 shard 20/20: selected 60 trajectories
v1_3_1: 1200 trajectories, 109200 agent-step rows

Saved v1.3.1 base: (109200, 17)
Trajectories: 1200
Shards: 20

Time-step statistics:
count    108000.000000
mean          0.100013
std           0.001534
min           0.058660
25%           0.099980
50%           0.099990
75%           0.100020
max           0.200910
Name: dt, dtype: float64


In [38]:
# EXTENSION 2A validation: WOMD v1.3.1 base
print("Shape:", df_131.shape)
print("Trajectories:", df_131["unit"].nunique())
print("Shards:", len(files_131))
counts_131 = df_131.groupby("unit")["step"].size()
print("\nSteps per trajectory:")
print(counts_131.describe())
print("\nDuplicate unit-step:", int(df_131.duplicated(["unit", "step"]).sum()))
print("Non-null dt:", int(df_131["dt"].notna().sum()))
print("Non-positive dt:", int((df_131["dt"].notna() & (df_131["dt"] <= 0)).sum()))
print("\ndt statistics:")
print(df_131["dt"].dropna().describe())
print(
    "\ndt outside 0.095–0.105 s:",
    int(((df_131["dt"].dropna() < 0.095) | (df_131["dt"].dropna() > 0.105)).sum()),
)
assert df_131.shape == (109200, 17)
assert df_131["unit"].nunique() == 1200
assert len(files_131) == 20
assert (counts_131 == 91).all()
assert not df_131.duplicated(["unit", "step"]).any()
assert df_131["dt"].notna().sum() == 108000
assert not (df_131["dt"].notna() & (df_131["dt"] <= 0)).any()
print("\nEXTENSION 2A validation: OK")

Shape: (109200, 17)
Trajectories: 1200
Shards: 20

Steps per trajectory:
count    1200.0
mean       91.0
std         0.0
min        91.0
25%        91.0
50%        91.0
75%        91.0
max        91.0
Name: step, dtype: float64

Duplicate unit-step: 0
Non-null dt: 108000
Non-positive dt: 0

dt statistics:
count    108000.000000
mean          0.100013
std           0.001534
min           0.058660
25%           0.099980
50%           0.099990
75%           0.100020
max           0.200910
Name: dt, dtype: float64

dt outside 0.095–0.105 s: 32

EXTENSION 2A validation: OK


In [39]:
traffic_131 = build_traffic_context(files_131, df_131["scenario"].unique())
df_131_traffic = merge_traffic_context(df_131, traffic_131)
df_131_traffic.to_parquet("data/processed/womd_v1_3_1_traffic.parquet", index=False)
print("Traffic rows:", traffic_131.shape)
print("Saved v1.3.1 + traffic:", df_131_traffic.shape)

2026-10-11 20:23:27.890413: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [1]
	 [[{{node Placeholder/_0}}]]
2026-10-11 20:23:32.588544: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [1]
	 [[{{node Placeholder/_0}}]]
2026-10-11 20:23:37.211552: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [1]
	 [[{{n

Traffic rows: (99827, 7)
Saved v1.3.1 + traffic: (109200, 21)


In [40]:
# EXTENSION 2B validation: v1.3.1 traffic context merge
n_scenarios_131 = df_131["scenario"].nunique()
print("Selected scenarios:", n_scenarios_131)
print("Traffic shape:", traffic_131.shape)
print("Merged shape:", df_131_traffic.shape)
print("\nTraffic scenarios:", traffic_131["scenario"].nunique())
print("Duplicate traffic scenario-step:", int(traffic_131.duplicated(["scenario", "step"]).sum()))
traffic_cols = ["n_traffic_lights", "n_stop_lights", "n_caution_lights", "n_go_lights"]
print("\nMissing traffic values:")
print(df_131_traffic[traffic_cols].isna().sum())
print("\nTraffic-light summary:")
print(df_131_traffic[traffic_cols].describe().round(3))
# Integrity checks
assert n_scenarios_131 == 1097
assert traffic_131.shape == (1097 * 91, 7)
assert traffic_131["scenario"].nunique() == 1097
assert not traffic_131.duplicated(["scenario", "step"]).any()
assert df_131_traffic.shape == (109200, 21)
assert len(df_131_traffic) == len(df_131)
assert not df_131_traffic[traffic_cols].isna().any().any()
assert (df_131_traffic[traffic_cols] >= 0).all().all()
print("\nEXTENSION 2B traffic validation: OK")

Selected scenarios: 1097
Traffic shape: (99827, 7)
Merged shape: (109200, 21)

Traffic scenarios: 1097
Duplicate traffic scenario-step: 0

Missing traffic values:
n_traffic_lights    0
n_stop_lights       0
n_caution_lights    0
n_go_lights         0
dtype: int64

Traffic-light summary:
       n_traffic_lights  n_stop_lights  n_caution_lights  n_go_lights
count        109200.000     109200.000        109200.000   109200.000
mean              6.658          3.456             0.202        1.480
std               6.597          4.128             0.973        2.413
min               0.000          0.000             0.000        0.000
25%               0.000          0.000             0.000        0.000
50%               6.000          2.000             0.000        0.000
75%              12.000          6.000             0.000        3.000
max              31.000         26.000            11.000       13.000

EXTENSION 2B traffic validation: OK


In [41]:
# EXTENSION 2C: build validated features for WOMD v1.3.1
feat_131 = build_feature_table(df_131, include_traffic=False)
feat_131_traffic = build_feature_table(df_131_traffic, include_traffic=True)
feat_131.to_parquet("data/processed/feat_v1_3_1.parquet", index=False)
feat_131_traffic.to_parquet("data/processed/feat_v1_3_1_traffic.parquet", index=False)
print("feat_v1_3_1:", feat_131.shape)
print("feat_v1_3_1_traffic:", feat_131_traffic.shape)
print("\nValid targets — base:")
print("3s:", feat_131["y_disp_3s"].notna().sum())
print("8s:", feat_131["y_disp_8s"].notna().sum())
print("\nValid targets — traffic:")
print("3s:", feat_131_traffic["y_disp_3s"].notna().sum())
print("8s:", feat_131_traffic["y_disp_8s"].notna().sum())

feat_v1_3_1: (109200, 20)
feat_v1_3_1_traffic: (109200, 24)

Valid targets — base:
3s: 72835
8s: 13031

Valid targets — traffic:
3s: 72835
8s: 13031


## EXTENSION 3 — WOMD v1.3.1 `sdc_paths` (tf.Example, SDC-only)

Không ép ghép tf.Example với Scenario sample. Dataset độc lập: một dòng cho mỗi scenario/SDC.


In [42]:
TFEXAMPLE_DIR = "data/raw/womd_v1_3_1_tfexample"
tf_files_131 = sorted(glob.glob(f"{TFEXAMPLE_DIR}/*tfrecord-*"))[:20]
assert len(tf_files_131) == 20
raw_record = next(iter(tf.data.TFRecordDataset(tf_files_131[0], compression_type="")))
example = tf.train.Example()
example.ParseFromString(raw_record.numpy())
keys = sorted(example.features.feature.keys())
path_keys = [k for k in keys if "path" in k.lower() or "sdc" in k.lower()]
print("tf.Example files:", len(tf_files_131))
print("Total feature keys:", len(keys))
for k in path_keys:
    f = example.features.feature[k]
    kind = f.WhichOneof("kind")
    vals = getattr(f, kind).value if kind else []
    print(f"{k:45s} | {kind:10s} | length={len(vals)}")

tf.Example files: 20
Total feature keys: 77
path_samples/arc_length                       | float_list | length=36000
path_samples/id                               | int64_list | length=36000
path_samples/on_route                         | int64_list | length=45
path_samples/valid                            | int64_list | length=36000
path_samples/xyz                              | float_list | length=108000
state/is_sdc                                  | int64_list | length=128


2026-10-11 20:24:55.716867: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [1]
	 [[{{node Placeholder/_0}}]]


In [43]:
# EXTENSION 3B: validate tf.Example dimensions before full SDC extraction
f = example.features.feature
expected_lengths = {
    # SDC / agent axis
    "state/is_sdc": 128,
    # Current state: 128 agents
    "state/current/valid": 128,
    "state/current/x": 128,
    "state/current/y": 128,
    "state/current/speed": 128,
    "state/current/bbox_yaw": 128,
    "state/current/length": 128,
    # Past: 128 agents × 10 steps
    "state/past/valid": 128 * 10,
    "state/past/speed": 128 * 10,
    "state/past/bbox_yaw": 128 * 10,
    # Future: 128 agents × 80 steps
    "state/future/valid": 128 * 80,
    "state/future/x": 128 * 80,
    "state/future/y": 128 * 80,
    # SDC paths: 45 paths × 800 points
    "path_samples/arc_length": 45 * 800,
    "path_samples/valid": 45 * 800,
    "path_samples/on_route": 45,
    # xyz = 45 × 800 × 3
    "path_samples/xyz": 45 * 800 * 3,
}
print("Required-field dimension check:\n")
for key, expected in expected_lengths.items():
    assert key in f, f"Missing required feature: {key}"
    feature = f[key]
    kind = feature.WhichOneof("kind")
    values = getattr(feature, kind).value
    actual = len(values)
    print(
        f"{key:35s} "
        f"actual={actual:6d} "
        f"expected={expected:6d} "
        f"{'OK' if actual == expected else 'MISMATCH'}"
    )
    assert actual == expected, f"{key}: expected {expected}, got {actual}"
# Scenario ID should contain exactly one value
assert "scenario/id" in f
assert len(f["scenario/id"].bytes_list.value) == 1
# Exactly one SDC is expected
is_sdc = np.array(f["state/is_sdc"].int64_list.value)
print("\nNumber of SDC slots:", int((is_sdc == 1).sum()))
assert (is_sdc == 1).sum() == 1
print("\nEXTENSION 3B schema validation: OK")

Required-field dimension check:

state/is_sdc                        actual=   128 expected=   128 OK
state/current/valid                 actual=   128 expected=   128 OK
state/current/x                     actual=   128 expected=   128 OK
state/current/y                     actual=   128 expected=   128 OK
state/current/speed                 actual=   128 expected=   128 OK
state/current/bbox_yaw              actual=   128 expected=   128 OK
state/current/length                actual=   128 expected=   128 OK
state/past/valid                    actual=  1280 expected=  1280 OK
state/past/speed                    actual=  1280 expected=  1280 OK
state/past/bbox_yaw                 actual=  1280 expected=  1280 OK
state/future/valid                  actual= 10240 expected= 10240 OK
state/future/x                      actual= 10240 expected= 10240 OK
state/future/y                      actual= 10240 expected= 10240 OK
path_samples/arc_length             actual= 36000 expected= 36000 OK
p

In [44]:
# EXTENSION 3C: dry-run SDC extraction on ONE tf.Example
N_AGENTS = 128
N_PAST = 10
N_FUTURE = 80
N_PATHS = 45
POINTS_PER_PATH = 800
f = example.features.feature
# 1. Identify SDC
sid = f["scenario/id"].bytes_list.value[0].decode("utf-8")
is_sdc = np.array(f["state/is_sdc"].int64_list.value)
slots = np.where(is_sdc == 1)[0]
assert len(slots) == 1
sdc = int(slots[0])
print("Scenario:", sid)
print("SDC slot:", sdc)
# 2. Current state
cur_valid = np.array(f["state/current/valid"].int64_list.value)
print("\nCurrent valid:", int(cur_valid[sdc]))
assert cur_valid[sdc] == 1
cur_x = np.array(f["state/current/x"].float_list.value)
cur_y = np.array(f["state/current/y"].float_list.value)
cur_speed = np.array(f["state/current/speed"].float_list.value)
cur_yaw = np.array(f["state/current/bbox_yaw"].float_list.value)
cur_length = np.array(f["state/current/length"].float_list.value)
x0 = cur_x[sdc]
y0 = cur_y[sdc]
speed0 = cur_speed[sdc]
yaw0 = cur_yaw[sdc]
length0 = cur_length[sdc]
print("Current x/y:", float(x0), float(y0))
print("Current speed:", float(speed0))
print("Current yaw:", float(yaw0))
print("Length:", float(length0))
# 3. Past state
past_speed = np.array(f["state/past/speed"].float_list.value).reshape(N_AGENTS, N_PAST)
past_yaw = np.array(f["state/past/bbox_yaw"].float_list.value).reshape(N_AGENTS, N_PAST)
past_valid = np.array(f["state/past/valid"].int64_list.value).reshape(N_AGENTS, N_PAST)
ps = past_speed[sdc]
py = past_yaw[sdc]
pv = past_valid[sdc]
print("\nValid past steps:", int(pv.sum()), "/", N_PAST)
if pv.sum() == N_PAST:
    dspeed_10step = float(speed0 - ps[0])
    # current speed minus immediately previous speed
    acc = float((speed0 - ps[-1]) / 0.1)
    yaw_series = np.unwrap(np.concatenate([py, [yaw0]]))
    yaw_rate_series = np.diff(yaw_series) / 0.1
    yaw_rate = float(yaw_rate_series[-1])
    yaw_ma1s = float(yaw_rate_series.mean())
    print("dspeed_10step:", dspeed_10step)
    print("acc:", acc)
    print("yaw_rate:", yaw_rate)
    print("yaw_ma1s:", yaw_ma1s)
else:
    print("Past history incomplete — this row would be skipped.")
# 4. Future targets
future_x = np.array(f["state/future/x"].float_list.value).reshape(N_AGENTS, N_FUTURE)
future_y = np.array(f["state/future/y"].float_list.value).reshape(N_AGENTS, N_FUTURE)
future_valid = np.array(f["state/future/valid"].int64_list.value).reshape(N_AGENTS, N_FUTURE)
fv = future_valid[sdc]
print("\nValid future steps:", int(fv.sum()), "/", N_FUTURE)
print("3s index 29 valid:", int(fv[29]))
print("8s index 79 valid:", int(fv[79]))
y_disp_3s = float(np.hypot(future_x[sdc, 29] - x0, future_y[sdc, 29] - y0)) if fv[29] == 1 else np.nan
y_disp_8s = float(np.hypot(future_x[sdc, 79] - x0, future_y[sdc, 79] - y0)) if fv[79] == 1 else np.nan
print("y_disp_3s:", y_disp_3s)
print("y_disp_8s:", y_disp_8s)
# 5. SDC paths
arc = np.array(f["path_samples/arc_length"].float_list.value).reshape(N_PATHS, POINTS_PER_PATH)
path_valid = np.array(f["path_samples/valid"].int64_list.value).reshape(N_PATHS, POINTS_PER_PATH)
on_route = np.array(f["path_samples/on_route"].int64_list.value)
valid_points = path_valid.sum(axis=1)
path_lengths = np.full(N_PATHS, np.nan)
for p in range(N_PATHS):
    idx = np.where(path_valid[p] == 1)[0]
    if len(idx) > 0:
        path_lengths[p] = arc[p, idx[-1]]
valid_mask = valid_points > 0
print("\nValid paths:", int(valid_mask.sum()), "/", N_PATHS)
print("On-route paths:", int(on_route.sum()))
if valid_mask.any():
    print(
        "Path length mean/min/max:",
        float(np.nanmean(path_lengths)),
        float(np.nanmin(path_lengths)),
        float(np.nanmax(path_lengths)),
    )
print("\nEXTENSION 3C dry-run: OK")

Scenario: 21fb0f7fd1ddda5
SDC slot: 5

Current valid: 1
Current x/y: 571.58447265625 1450.7574462890625
Current speed: 20.145776748657227
Current yaw: 3.134042978286743
Length: 5.285999774932861

Valid past steps: 10 / 10
dspeed_10step: -0.06155967712402344
acc: 0.006580352783203125
yaw_rate: -0.003979206085205078
yaw_ma1s: -0.0010728836059570312

Valid future steps: 80 / 80
3s index 29 valid: 1
8s index 79 valid: 1
y_disp_3s: 60.22624199492859
y_disp_8s: 160.99106746851558

Valid paths: 45 / 45
On-route paths: 30
Path length mean/min/max: 584.6399563259548 315.9638977050781 776.7921142578125

EXTENSION 3C dry-run: OK


In [45]:
# EXTENSION 3D: full SDC-path extraction from 20 tf.Example shards
N_AGENTS = 128
N_PAST = 10
N_FUTURE = 80
N_PATHS = 45
POINTS_PER_PATH = 800
rows_sdc = []
skip_bad_sdc = 0
skip_current_invalid = 0
skip_past_incomplete = 0
for file_i, fpath in enumerate(tf_files_131):
    print(f"Processing tf.Example file " f"{file_i + 1}/{len(tf_files_131)}")
    for raw in tf.data.TFRecordDataset(fpath, compression_type=""):
        ex = tf.train.Example()
        ex.ParseFromString(raw.numpy())
        f = ex.features.feature
        # Scenario / SDC
        sid = f["scenario/id"].bytes_list.value[0].decode("utf-8")
        is_sdc = np.array(f["state/is_sdc"].int64_list.value)
        slots = np.where(is_sdc == 1)[0]
        if len(slots) != 1:
            skip_bad_sdc += 1
            continue
        sdc = int(slots[0])
        # Current state
        cur_valid = np.array(f["state/current/valid"].int64_list.value)
        if cur_valid[sdc] != 1:
            skip_current_invalid += 1
            continue
        cur_x = np.array(f["state/current/x"].float_list.value)
        cur_y = np.array(f["state/current/y"].float_list.value)
        cur_speed = np.array(f["state/current/speed"].float_list.value)
        cur_yaw = np.array(f["state/current/bbox_yaw"].float_list.value)
        cur_length = np.array(f["state/current/length"].float_list.value)
        x0 = cur_x[sdc]
        y0 = cur_y[sdc]
        speed0 = cur_speed[sdc]
        yaw0 = cur_yaw[sdc]
        length0 = cur_length[sdc]
        n_agents = int(cur_valid.sum())
        # Past state
        past_speed = np.array(f["state/past/speed"].float_list.value).reshape(N_AGENTS, N_PAST)
        past_yaw = np.array(f["state/past/bbox_yaw"].float_list.value).reshape(N_AGENTS, N_PAST)
        past_valid = np.array(f["state/past/valid"].int64_list.value).reshape(N_AGENTS, N_PAST)
        ps = past_speed[sdc]
        py = past_yaw[sdc]
        pv = past_valid[sdc]
        if pv.sum() < N_PAST:
            skip_past_incomplete += 1
            continue
        # Same naming convention as Core
        dspeed_10step = float(speed0 - ps[0])
        # Immediate previous step -> current
        acc = float((speed0 - ps[-1]) / 0.1)
        yaw_series = np.unwrap(np.concatenate([py, [yaw0]]))
        yaw_rate_series = np.diff(yaw_series) / 0.1
        yaw_rate = float(yaw_rate_series[-1])
        yaw_ma1s = float(yaw_rate_series.mean())
        # Future targets
        # tf.Example future index 29 = 3 s
        # and index 79 = 8 s.
        # Keep NaN when unavailable; clean later.
        future_x = np.array(f["state/future/x"].float_list.value).reshape(N_AGENTS, N_FUTURE)
        future_y = np.array(f["state/future/y"].float_list.value).reshape(N_AGENTS, N_FUTURE)
        future_valid = np.array(f["state/future/valid"].int64_list.value).reshape(N_AGENTS, N_FUTURE)
        fv = future_valid[sdc]
        y_disp_3s = float(np.hypot(future_x[sdc, 29] - x0, future_y[sdc, 29] - y0)) if fv[29] == 1 else np.nan
        y_disp_8s = float(np.hypot(future_x[sdc, 79] - x0, future_y[sdc, 79] - y0)) if fv[79] == 1 else np.nan
        # SDC paths
        arc = np.array(f["path_samples/arc_length"].float_list.value).reshape(N_PATHS, POINTS_PER_PATH)
        path_valid = np.array(f["path_samples/valid"].int64_list.value).reshape(N_PATHS, POINTS_PER_PATH)
        on_route = np.array(f["path_samples/on_route"].int64_list.value)
        valid_points = path_valid.sum(axis=1)
        path_lengths = np.full(N_PATHS, np.nan)
        for p in range(N_PATHS):
            idx = np.where(path_valid[p] == 1)[0]
            if len(idx) > 0:
                path_lengths[p] = arc[p, idx[-1]]
        valid_mask = valid_points > 0
        if valid_mask.any():
            pl_mean = float(np.nanmean(path_lengths))
            pl_min = float(np.nanmin(path_lengths))
            pl_max = float(np.nanmax(path_lengths))
        else:
            pl_mean = np.nan
            pl_min = np.nan
            pl_max = np.nan
        # Store one row per scenario / SDC
        rows_sdc.append(
            {
                "scenario": sid,
                "speed": float(speed0),
                "acc": acc,
                "yaw_rate": yaw_rate,
                "dspeed_10step": dspeed_10step,
                "yaw_ma1s": yaw_ma1s,
                "length": float(length0),
                "n_agents": n_agents,
                "num_valid_paths": int(valid_mask.sum()),
                "num_on_route_paths": int(on_route.sum()),
                "path_length_mean": pl_mean,
                "path_length_min": pl_min,
                "path_length_max": pl_max,
                "y_disp_3s": y_disp_3s,
                "y_disp_8s": y_disp_8s,
            }
        )
# Raw extracted table
sdc_df = pd.DataFrame(rows_sdc)
print("\nSDC raw dataset:", sdc_df.shape)
print("Unique scenarios:", sdc_df["scenario"].nunique())
print("\nSkipped records:")
print("bad SDC slot       =", skip_bad_sdc)
print("current invalid    =", skip_current_invalid)
print("past incomplete    =", skip_past_incomplete)
print("\nTarget availability:")
print("Valid 3s:", sdc_df["y_disp_3s"].notna().sum())
print("Valid 8s:", sdc_df["y_disp_8s"].notna().sum())
print("\nPath availability:")
print("Rows with 0 valid paths:", int((sdc_df["num_valid_paths"] == 0).sum()))
print("\nMissing values:")
print(sdc_df.isna().sum().sort_values(ascending=False))
print("\nDuplicate scenarios:", int(sdc_df["scenario"].duplicated().sum()))
display(sdc_df.head())

Processing tf.Example file 1/20


2026-10-11 20:24:55.928281: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [1]
	 [[{{node Placeholder/_0}}]]


Processing tf.Example file 2/20


2026-10-11 20:25:07.949197: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [1]
	 [[{{node Placeholder/_0}}]]


Processing tf.Example file 3/20


...

2026-10-11 20:28:25.895053: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [1]
	 [[{{node Placeholder/_0}}]]


Processing tf.Example file 20/20


2026-10-11 20:28:37.608822: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [1]
	 [[{{node Placeholder/_0}}]]



SDC raw dataset: (9466, 15)
Unique scenarios: 9466

Skipped records:
bad SDC slot       = 0
current invalid    = 0
past incomplete    = 0

Target availability:
Valid 3s: 9466
Valid 8s: 9466

Path availability:
Rows with 0 valid paths: 40

Missing values:
path_length_mean      40
path_length_min       40
path_length_max       40
scenario               0
speed                  0
acc                    0
yaw_rate               0
dspeed_10step          0
yaw_ma1s               0
length                 0
n_agents               0
num_valid_paths        0
num_on_route_paths     0
y_disp_3s              0
y_disp_8s              0
dtype: int64

Duplicate scenarios: 0


,scenario,speed,acc,yaw_rate,dspeed_10step,yaw_ma1s,length,n_agents,num_valid_paths,num_on_route_paths,path_length_mean,path_length_min,path_length_max,y_disp_3s,y_disp_8s
0,21fb0f7fd1ddda5,20.145777,0.006580,-0.003979,-0.061560,-0.001073,5.286,9,45,30,584.639956,315.963898,776.792114,60.226242,160.991067
1,fd83fb20bf28325e,6.728274,0.044212,-0.006757,0.073405,-0.000517,5.286,40,15,15,548.589106,205.558853,770.723755,20.114305,49.813295
2,274c53d87aa7c3ed,19.358414,0.222664,0.000759,0.153296,0.000032,5.286,14,4,4,451.889496,365.788940,484.425171,57.551434,137.899914
3,8e5412c087e2e8f4,0.000987,0.002805,-0.000012,0.000402,0.000000,5.286,30,3,3,178.085770,144.376984,240.896408,0.000122,0.000546
4,104a523e50e228c9,2.508610,-0.091491,0.051067,0.169457,0.020476,5.286,20,1,0,209.462814,209.462814,209.462814,6.811263,34.168960


In [46]:
# EXTENSION 3E: clean and save SDC-path dataset
path_length_cols = ["path_length_mean", "path_length_min", "path_length_max"]
# 1. Verify that path-length NaN occurs ONLY
#    when the scenario has zero valid paths
no_valid_path = sdc_df["num_valid_paths"] == 0
missing_path = sdc_df[path_length_cols].isna().any(axis=1)
print("Rows with zero valid paths:", int(no_valid_path.sum()))
print("Rows with missing path length:", int(missing_path.sum()))
assert missing_path.equals(no_valid_path)
# 2. Verify targets and identifiers before cleaning
assert sdc_df["y_disp_3s"].notna().all()
assert sdc_df["y_disp_8s"].notna().all()
assert not sdc_df["scenario"].duplicated().any()
# 3. Clean
sdc_clean = sdc_df.copy()
# Explicit indicator:
# 1 = at least one valid candidate SDC path
# 0 = no valid candidate path available
sdc_clean["has_valid_path"] = (~no_valid_path).astype("int8")
# Path-length summaries are undefined when no paths exist.
# Fill with 0 only together with has_valid_path / num_valid_paths,
# so downstream models can distinguish "no path" from valid paths.
sdc_clean[path_length_cols] = sdc_clean[path_length_cols].fillna(0.0)
# 4. Final integrity checks
assert sdc_clean.isna().sum().sum() == 0
assert not sdc_clean["scenario"].duplicated().any()
assert len(sdc_clean) == 9466
assert (sdc_clean.loc[sdc_clean["has_valid_path"] == 0, "num_valid_paths"] == 0).all()
assert (sdc_clean.loc[sdc_clean["has_valid_path"] == 0, path_length_cols] == 0).all().all()
# 5. Save
sdc_clean.to_parquet("data/processed/sdc_v1_3_1_paths.parquet", index=False)
print("\nSDC clean shape:", sdc_clean.shape)
print("Zero-valid-path rows:", int(no_valid_path.sum()))
print("Rows with valid paths:", int(sdc_clean["has_valid_path"].sum()))
print("Total missing after cleaning:", int(sdc_clean.isna().sum().sum()))
print("Duplicate scenarios:", int(sdc_clean["scenario"].duplicated().sum()))
print("\nEXTENSION 3E cleaning: OK")

Rows with zero valid paths: 40
Rows with missing path length: 40

SDC clean shape: (9466, 16)
Zero-valid-path rows: 40
Rows with valid paths: 9426
Total missing after cleaning: 0
Duplicate scenarios: 0

EXTENSION 3E cleaning: OK


### EXTENSION 3F: ghi rõ xe nào được dự báo và mỗi biến được biết từ lúc nào (góp ý 5.3)

- **Xe được dự báo là chính xe tự lái (SDC)**, mỗi cảnh đúng một dòng. Đây không phải mẫu 1,200 xe của CORE.
- **Thời điểm dự báo:** chỉ số `current` của tf.Example, tức t = 1.0 s (sau 10 bước quá khứ). Đây trùng với thời điểm step 10 của CORE.
- **Mục tiêu:** vị trí tương lai tại index 29 (3 s) và 79 (8 s) của `state/future`.
- **Khác với CORE:** gia tốc tính qua một bước (dt = 0.1 s cố định); `yaw_ma1s` tính bằng `np.unwrap`; không có `acc_ma1s`; `length` gần như là hằng số vì mọi SDC cùng một loại xe.
- **Rủi ro rò rỉ cần kiểm tra:** `path_samples/on_route` cho biết đường đi ứng viên có nằm trên lộ trình (route) của SDC không. Nhóm phải đọc tài liệu WOMD v1.3.1 để xác nhận lộ trình là thông tin **lập kế hoạch có sẵn tại t = 1.0 s**, hay được suy ra từ **đường SDC thực sự đã đi**. Nếu là trường hợp sau thì `num_on_route_paths` là xem trước đáp án. Ô dưới lưu thêm bản dữ liệu **không có** biến này để Notebook B chạy ablation: nếu lợi ích ở 8 s biến mất khi bỏ biến này, bài phải ghi rõ.

In [47]:
# EXTENSION 3F (docx 5.3): document the forecast agent / information timing, and save a no-route variant
sdc_doc = pd.read_parquet("data/processed/sdc_v1_3_1_paths.parquet")
# Same physical clipping as CORE so the definitions do not diverge further
if CLIP_KINEMATICS:
    n_acc = int((sdc_doc["acc"].abs() > ACC_LIMIT).sum())
    n_yaw = int((sdc_doc["yaw_rate"].abs() > YAW_RATE_LIMIT).sum())
    sdc_doc["acc"] = sdc_doc["acc"].clip(-ACC_LIMIT, ACC_LIMIT)
    sdc_doc["yaw_rate"] = sdc_doc["yaw_rate"].clip(-YAW_RATE_LIMIT, YAW_RATE_LIMIT)
    sdc_doc["yaw_ma1s"] = sdc_doc["yaw_ma1s"].clip(-YAW_RATE_LIMIT, YAW_RATE_LIMIT)
    CLIP_LOG["v1_3_1_sdc"] = {"rows": len(sdc_doc), "acc_clipped": n_acc, "yaw_rate_clipped": n_yaw}
    print("SDC clipping:", CLIP_LOG["v1_3_1_sdc"])
sdc_doc["forecast_agent"] = "SDC"
sdc_doc["origin_step"] = ORIGIN_STEP
sdc_doc["origin_time_s"] = 1.0
sdc_doc.to_parquet("data/processed/sdc_v1_3_1_paths.parquet", index=False)
sdc_noroute = sdc_doc.drop(columns=["num_on_route_paths"])
sdc_noroute.to_parquet("data/processed/sdc_v1_3_1_paths_noroute.parquet", index=False)
timing = pd.DataFrame(
    [
        ["speed, acc, yaw_rate, dspeed_10step, yaw_ma1s", "state/current + state/past", "t <= 1.0 s", "safe"],
        ["length", "state/current/length", "t = 1.0 s", "safe (almost constant for SDC)"],
        ["n_agents", "state/current/valid (all 128 slots, all object types)", "t = 1.0 s", "safe"],
        [
            "num_valid_paths, path_length_mean/min/max",
            "path_samples/valid, arc_length",
            "stated by WOMD as candidate paths from the SDC position",
            "verify in WOMD docs",
        ],
        [
            "num_on_route_paths",
            "path_samples/on_route",
            "depends on how WOMD defines the SDC route",
            "POSSIBLE LEAKAGE: verify; run ablation without it",
        ],
        [
            "y_disp_3s / y_disp_8s (targets)",
            "state/future/x,y index 29 / 79",
            "t = 4.0 s / 9.0 s",
            "target only",
        ],
    ],
    columns=["feature", "source field", "when known", "status"],
)
timing.to_csv("report/table_sdc_feature_timing.csv", index=False)
display(timing)
print(
    "Forecast agent: SDC | rows (= scenarios):",
    len(sdc_doc),
    "| distinct SDC length values:",
    sdc_doc["length"].nunique(),
)
print("Share of rows with >=1 on-route path:", round(float((sdc_doc["num_on_route_paths"] > 0).mean()), 4))
print("Saved: sdc_v1_3_1_paths.parquet (with documentation columns) and sdc_v1_3_1_paths_noroute.parquet")

SDC clipping: {'rows': 9466, 'acc_clipped': 0, 'yaw_rate_clipped': 0}


,feature,source field,when known,status
0,"speed, acc, yaw_rate, dspeed_10step, yaw_ma1s",state/current + state/past,t <= 1.0 s,safe
1,length,state/current/length,t = 1.0 s,safe (almost constant for SDC)
2,n_agents,"state/current/valid (all 128 slots, all object...",t = 1.0 s,safe
3,"num_valid_paths, path_length_mean/min/max","path_samples/valid, arc_length",stated by WOMD as candidate paths from the SDC...,verify in WOMD docs
4,num_on_route_paths,path_samples/on_route,depends on how WOMD defines the SDC route,POSSIBLE LEAKAGE: verify; run ablation without it
5,y_disp_3s / y_disp_8s (targets),"state/future/x,y index 29 / 79",t = 4.0 s / 9.0 s,target only


Forecast agent: SDC | rows (= scenarios): 9466 | distinct SDC length values: 1
Share of rows with >=1 on-route path: 0.9709
Saved: sdc_v1_3_1_paths.parquet (with documentation columns) and sdc_v1_3_1_paths_noroute.parquet


## Final validation for extension datasets

Không ghi đè file core.


In [48]:
# FINAL VALIDATION — all Notebook A extension datasets
# Also verify that the validated Core handoff was not overwritten.
import hashlib
import json
import os
TRAFFIC_COLS = ["n_traffic_lights", "n_stop_lights", "n_caution_lights", "n_go_lights"]
EXPECTED_TARGETS = {"v1_3_0": {"3s": 72835, "8s": 13031}, "v1_3_1": {"3s": 72835, "8s": 13031}}
# 1. Verify CORE has NOT been overwritten
with open("data/processed/manifest.json", "r") as f:
    core_manifest = json.load(f)
with open("data/processed/feat.parquet", "rb") as f:
    current_core_md5 = hashlib.md5(f.read()).hexdigest()
print("Core manifest version:", core_manifest["dataset_version"])
print("Core manifest rows:", core_manifest["rows"])
print("Core MD5 unchanged:", current_core_md5 == core_manifest["md5"])
assert core_manifest["dataset_version"] == "WOMD v1.2"
assert core_manifest["rows"] == EXPECTED_CORE["rows"]
with open("data/processed/feat_core_origin10.parquet", "rb") as f:
    assert (
        hashlib.md5(f.read()).hexdigest() == core_manifest["origin_md5"]
    ), "origin table changed after handoff"
print("Core origin table MD5 unchanged: True")
o_b = pd.read_parquet("data/processed/feat_v1_3_0_origin10.parquet")
o_t = pd.read_parquet("data/processed/feat_v1_3_0_traffic_origin10.parquet")
assert o_b[["scenario", "unit", "step"]].equals(o_t[["scenario", "unit", "step"]])
assert (o_b["step"] == ORIGIN_STEP).all() and not o_b.duplicated("unit").any()
print("v1.3.0 origin tables:", o_b.shape, o_t.shape)
assert current_core_md5 == core_manifest["md5"]
# 2. Validate v1.3.0 and v1.3.1 base / traffic
for version in ["v1_3_0", "v1_3_1"]:
    print("\n" + "=" * 60)
    print(version)
    print("=" * 60)
    b = (
        pd.read_parquet(f"data/processed/feat_{version}.parquet")
        .sort_values(["scenario", "unit", "step"])
        .reset_index(drop=True)
    )
    t = (
        pd.read_parquet(f"data/processed/feat_{version}_traffic.parquet")
        .sort_values(["scenario", "unit", "step"])
        .reset_index(drop=True)
    )
    print("Base shape   :", b.shape)
    print("Traffic shape:", t.shape)
    assert b.shape == (109200, 20)
    assert t.shape == (109200, 24)
    assert b["unit"].nunique() == 1200
    assert t["unit"].nunique() == 1200
    assert not b.duplicated(["unit", "step"]).any()
    assert not t.duplicated(["unit", "step"]).any()
    # Base and traffic must contain same trajectories
    assert b[["scenario", "unit", "step"]].equals(t[["scenario", "unit", "step"]])
    # Traffic version should add ONLY traffic columns
    added_cols = [c for c in t.columns if c not in b.columns]
    print("Added traffic columns:", added_cols)
    assert set(added_cols) == set(TRAFFIC_COLS)
    for col in TRAFFIC_COLS:
        assert col in t.columns
        assert t[col].notna().all()
        assert (t[col] >= 0).all()
    # Core trajectory information must be identical
    for col in [
        "ts",
        "x",
        "y",
        "vx",
        "vy",
        "heading",
        "length",
        "width",
        "n_agents",
        "speed",
        "acc",
        "yaw_rate",
        "dspeed_10step",
        "acc_ma1s",
        "yaw_ma1s",
        "y_disp_3s",
        "y_disp_8s",
    ]:
        assert np.allclose(b[col], t[col], equal_nan=True), f"{version}: mismatch in {col}"
    # Validated targets
    n3_base = int(b["y_disp_3s"].notna().sum())
    n8_base = int(b["y_disp_8s"].notna().sum())
    n3_traffic = int(t["y_disp_3s"].notna().sum())
    n8_traffic = int(t["y_disp_8s"].notna().sum())
    print("Valid 3s:", n3_base)
    print("Valid 8s:", n8_base)
    assert n3_base == EXPECTED_TARGETS[version]["3s"]
    assert n8_base == EXPECTED_TARGETS[version]["8s"]
    assert n3_base == n3_traffic
    assert n8_base == n8_traffic
    # No leaked future helper columns
    future_helper_cols = [
        c
        for c in t.columns
        if any(token in c for token in ["lead30", "lead80", "ts_lead", "x_lead", "y_lead"])
    ]
    assert future_helper_cols == []
    print(f"{version} base/traffic validation: OK")
# 3. Validate v1.3.1 SDC-path dataset
print("\n" + "=" * 60)
print("v1.3.1 SDC paths")
print("=" * 60)
sdc_check = pd.read_parquet("data/processed/sdc_v1_3_1_paths.parquet")
print("Shape:", sdc_check.shape)
print("Unique scenarios:", sdc_check["scenario"].nunique())
print("Missing values:", int(sdc_check.isna().sum().sum()))
print("Duplicate scenarios:", int(sdc_check["scenario"].duplicated().sum()))
print("Valid 3s targets:", int(sdc_check["y_disp_3s"].notna().sum()))
print("Valid 8s targets:", int(sdc_check["y_disp_8s"].notna().sum()))
print("Rows with valid paths:", int(sdc_check["has_valid_path"].sum()))
print("Rows without valid paths:", int((sdc_check["has_valid_path"] == 0).sum()))
# Structural checks
assert sdc_check.shape == (9466, 19)
assert (sdc_check["forecast_agent"] == "SDC").all()
assert "num_on_route_paths" not in pd.read_parquet("data/processed/sdc_v1_3_1_paths_noroute.parquet").columns
assert sdc_check["scenario"].nunique() == 9466
assert not (sdc_check["scenario"].duplicated().any())
assert sdc_check.isna().sum().sum() == 0
assert sdc_check["y_disp_3s"].notna().sum() == 9466
assert sdc_check["y_disp_8s"].notna().sum() == 9466
assert sdc_check["has_valid_path"].sum() == 9426
assert (sdc_check["has_valid_path"] == 0).sum() == 40
assert (sdc_check.loc[sdc_check["has_valid_path"] == 0, "num_valid_paths"] == 0).all()
path_length_cols = ["path_length_mean", "path_length_min", "path_length_max"]
assert (sdc_check.loc[sdc_check["has_valid_path"] == 0, path_length_cols] == 0).all().all()
print("v1.3.1 SDC-path validation: OK")
print("\nALL NOTEBOOK A EXTENSION TESTS: OK")

Core manifest version: WOMD v1.2
Core manifest rows: 109200
Core MD5 unchanged: True
Core origin table MD5 unchanged: True
v1.3.0 origin tables: (1187, 23) (1187, 27)

v1_3_0
Base shape   : (109200, 20)
Traffic shape: (109200, 24)
Added traffic columns: ['n_traffic_lights', 'n_stop_lights', 'n_caution_lights', 'n_go_lights']
Valid 3s: 72835
Valid 8s: 13031
v1_3_0 base/traffic validation: OK

v1_3_1
Base shape   : (109200, 20)
Traffic shape: (109200, 24)
Added traffic columns: ['n_traffic_lights', 'n_stop_lights', 'n_caution_lights', 'n_go_lights']
Valid 3s: 72835
Valid 8s: 13031
v1_3_1 base/traffic validation: OK

v1.3.1 SDC paths
Shape: (9466, 19)
Unique scenarios: 9466
Missing values: 0
Duplicate scenarios: 0
Valid 3s targets: 9466
Valid 8s targets: 9466
Rows with valid paths: 9426
Rows without valid paths: 40
v1.3.1 SDC-path validation: OK

ALL NOTEBOOK A EXTENSION TESTS: OK


In [49]:
# FINAL NOTEBOOK A OUTPUT AUDIT
# Only inspect files — do not delete anything yet.
import os
required_a = [
    # Core handoff
    "data/processed/feat.parquet",
    "data/processed/manifest.json",
    "data/processed/feat_v1_2.parquet",
    "data/processed/feat_core_origin10.parquet",
    # Review fixes (docx 1a, 3.3, 5.3, 5.4, last section)
    "report/table_sampling_v1_2.csv",
    "report/table_sampling_summary_v1_2.csv",
    "report/table_zero_displacement.csv",
    "report/table_core_origin_audit.csv",
    "report/table_core_origin_motion.csv",
    "report/table_rq1_stats.csv",
    "report/fig_rq1_caption.txt",
    "report/table_v1_3_0_origin_audit.csv",
    "data/processed/feat_v1_3_0_origin10.parquet",
    "data/processed/feat_v1_3_0_traffic_origin10.parquet",
    "data/processed/sdc_v1_3_1_paths_noroute.parquet",
    "report/table_sdc_feature_timing.csv",
    # Core EDA / audit
    "report/profile.html",
    "report/table_cleaning_log.csv",
    "report/table_describe_raw.csv",
    "report/table_describe_numeric.csv",
    "report/table_describe_categorical.csv",
    "report/table_eda_notes.csv",
    "report/table_missing.csv",
    "report/table_columns.csv",
    "report/table_corr.csv",
    # Validated targets
    "report/table_target_summary.csv",
    "report/table_target_by_unit.csv",
    "report/table_target_by_step.csv",
    # SQL
    "sql/queries.sql",
    "report/table_q1.csv",
    "report/table_q2.csv",
    "report/table_q3.csv",
    # RQ1
    "report/table_rq1_density.csv",
    "report/table_rq1_turning.csv",
    "report/table_rq1_density_unit.csv",
    "report/fig_rq1_density_turning.png",
    # v1.3.0 extension
    "data/processed/womd_v1_3_0.parquet",
    "data/processed/womd_v1_3_0_traffic.parquet",
    "data/processed/feat_v1_3_0.parquet",
    "data/processed/feat_v1_3_0_traffic.parquet",
    # v1.3.1 extension
    "data/processed/womd_v1_3_1.parquet",
    "data/processed/womd_v1_3_1_traffic.parquet",
    "data/processed/feat_v1_3_1.parquet",
    "data/processed/feat_v1_3_1_traffic.parquet",
    # SDC paths
    "data/processed/sdc_v1_3_1_paths.parquet",
]
old_or_stale = [
    "report/table_target_by_period.csv",
    "report/table_rq1a_period.csv",
    "report/table_rq1b_corr.csv",
    "report/table_rq1c_units.csv",
]
print("=== REQUIRED NOTEBOOK A FILES ===")
missing = []
for p in required_a:
    ok = os.path.exists(p)
    print(f"{'OK     ' if ok else 'MISSING'}  {p}")
    if not ok:
        missing.append(p)
print("\n=== POSSIBLE STALE FILES ===")
for p in old_or_stale:
    print(f"{'EXISTS ' if os.path.exists(p) else 'ABSENT '}  {p}")
print("\nRequired files:", len(required_a))
print("Missing required:", len(missing))
if missing:
    print("\nMissing:")
    for p in missing:
        print("-", p)
else:
    print("\nNOTEBOOK A OUTPUT AUDIT: OK")

=== REQUIRED NOTEBOOK A FILES ===
OK       data/processed/feat.parquet
OK       data/processed/manifest.json
OK       data/processed/feat_v1_2.parquet
OK       data/processed/feat_core_origin10.parquet
OK       report/table_sampling_v1_2.csv
OK       report/table_sampling_summary_v1_2.csv
OK       report/table_zero_displacement.csv
OK       report/table_core_origin_audit.csv
OK       report/table_core_origin_motion.csv
OK       report/table_rq1_stats.csv
OK       report/fig_rq1_caption.txt
OK       report/table_v1_3_0_origin_audit.csv
OK       data/processed/feat_v1_3_0_origin10.parquet
OK       data/processed/feat_v1_3_0_traffic_origin10.parquet
OK       data/processed/sdc_v1_3_1_paths_noroute.parquet
OK       report/table_sdc_feature_timing.csv
OK       report/profile.html
OK       report/table_cleaning_log.csv
OK       report/table_describe_raw.csv
OK       report/table_describe_numeric.csv
OK       report/table_describe_categorical.csv
OK       report/table_eda_notes.csv
OK       r

In [50]:
# FINAL CLEANUP A1: archive stale outputs from old Notebook A logic
import os
import shutil
stale_files = [
    "report/table_target_by_period.csv",
    "report/table_rq1a_period.csv",
    "report/table_rq1b_corr.csv",
    "report/table_rq1c_units.csv",
]
archive_dir = "report/_stale_notebook_a"
os.makedirs(archive_dir, exist_ok=True)
for src in stale_files:
    if os.path.exists(src):
        dst = os.path.join(archive_dir, os.path.basename(src))
        shutil.move(src, dst)
        print("MOVED:", src, "->", dst)
    else:
        print("ALREADY ABSENT:", src)
print("\nCurrent report files:")
for name in sorted(os.listdir("report")):
    if not name.startswith("_"):
        print(name)
print("\nFINAL CLEANUP A1: OK")

ALREADY ABSENT: report/table_target_by_period.csv
ALREADY ABSENT: report/table_rq1a_period.csv
ALREADY ABSENT: report/table_rq1b_corr.csv
ALREADY ABSENT: report/table_rq1c_units.csv

Current report files:
fig_rq1_caption.txt
fig_rq1_density_turning.pdf
fig_rq1_density_turning.png
missing_sentence.txt
profile.html
rq1_text_option.txt
table_cleaning_log.csv
table_columns.csv
table_core_origin_audit.csv
table_core_origin_motion.csv
table_corr.csv
table_describe_categorical.csv
table_describe_numeric.csv
table_describe_raw.csv
table_eda_notes.csv
table_missing.csv
table_q1.csv
table_q2.csv
table_q3.csv
table_rq1_density.csv
table_rq1_density_unit.csv
table_rq1_stats.csv
table_rq1_turning.csv
table_sampling_summary_v1_2.csv
table_sampling_v1_2.csv
table_sdc_feature_timing.csv
table_target_by_step.csv
table_target_by_unit.csv
table_target_summary.csv
table_v1_3_0_origin_audit.csv
table_zero_displacement.csv

FINAL CLEANUP A1: OK


## NUMBERS FOR PAPER (góp ý mục cuối)

Ô này in **mọi con số Notebook A cung cấp cho bài** và lưu vào `report/paper_numbers_A.csv` và `.json`. Khi viết bài chỉ chép số từ đây, không gõ theo trí nhớ. Mục 3.2 (thống kê mô tả) và mục 5.1 (RQ1) phải lấy số từ ô này, **không** lấy từ Notebook C.

In [51]:
# NUMBERS FOR PAPER — Notebook A
import json, os
P = {}
def put(section, key, value):
    if isinstance(value, (np.integer,)):
        value = int(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    P[f"{section}.{key}"] = value
core_df = pd.read_parquet("data/processed/womd_v1_2.parquet")
feat_all = pd.read_parquet("data/processed/feat_v1_2.parquet")
origin = pd.read_parquet("data/processed/feat_core_origin10.parquet")
# --- Section 3.1: sampling (docx 5.4) ---
samp = pd.read_csv("report/table_sampling_summary_v1_2.csv", index_col=0)["value"]
for k, v in samp.items():
    put("3.1_sampling", k, v)
# --- Section 3.1/3.2: data size and preprocessing ---
put("3.1_data", "rows", len(core_df))
put("3.1_data", "trajectories", core_df["unit"].nunique())
put("3.1_data", "scenarios", core_df["scenario"].nunique())
put("3.1_data", "valid_acc_yaw_rows", int(core_df["acc"].notna().sum()))
for k, v in CLIP_LOG.get("v1_2", {}).items():
    put("3.1_clipping", k, v)
# --- Section 3.2: descriptive statistics (replace the old Notebook C numbers) ---
for col in ["speed", "acc", "yaw_rate", "length", "n_agents"]:
    s = core_df[col].dropna()
    put("3.2_describe", f"{col}_mean", round(s.mean(), 4))
    put("3.2_describe", f"{col}_sd", round(s.std(), 4))
    put("3.2_describe", f"{col}_median", round(s.median(), 4))
    put("3.2_describe", f"{col}_min", round(s.min(), 4))
    put("3.2_describe", f"{col}_max", round(s.max(), 4))
# --- Section 3.2: targets ---
put("3.2_targets", "possible_3s", N_TRAJECTORIES * 61)
put("3.2_targets", "valid_3s", int(feat_all["y_disp_3s"].notna().sum()))
put("3.2_targets", "possible_8s", N_TRAJECTORIES * 11)
put("3.2_targets", "valid_8s", int(feat_all["y_disp_8s"].notna().sum()))
zt = pd.read_csv("report/table_zero_displacement.csv").set_index("target")
put("3.2_targets", "zero_share_3s", round(zt.loc["y_disp_3s", "zero_share"], 4))
put("3.2_targets", "zero_share_8s", round(zt.loc["y_disp_8s", "zero_share"], 4))
# --- Section 4.1: single-origin modelling table (docx 1a, 3.3, 5.5) ---
put("4.1_origin", "origin_step", ORIGIN_STEP)
put("4.1_origin", "rows", len(origin))
put("4.1_origin", "vehicles", origin["unit"].nunique())
put("4.1_origin", "scenarios", origin["scenario"].nunique())
put("4.1_origin", "n_features", len(COMMON_X))
put("4.1_origin", "stationary_share", round((origin["motion_state"] == "stationary").mean(), 4))
put("4.1_origin", "cv_MAE_3s_all", round((origin["y_disp_3s"] - origin["speed"] * 3).abs().mean(), 4))
put("4.1_origin", "cv_MAE_8s_all", round((origin["y_disp_8s"] - origin["speed"] * 8).abs().mean(), 4))
om = pd.read_csv("report/table_core_origin_motion.csv").set_index("motion_state")
for state in om.index:
    for c in ["vehicles", "share", "cv_MAE_3s", "cv_MAE_8s", "zero_disp_share_3s", "zero_disp_share_8s"]:
        put("4.1_origin", f"{state}_{c}", round(float(om.loc[state, c]), 4))
# --- Section 5.1: RQ1 (replace the old Notebook C numbers and Fig. 2) ---
rs = pd.read_csv("report/table_rq1_stats.csv").set_index("statistic")["value"]
for k, v in rs.items():
    put("5.1_rq1", k, v)
dt_ = pd.read_csv("report/table_rq1_density.csv").set_index("density_group")
for g in dt_.index:
    put("5.1_rq1", f"{g}_mean_speed", round(dt_.loc[g, "mean_speed"], 4))
    put("5.1_rq1", f"{g}_event_share", round(dt_.loc[g, "event_share"], 4))
# --- Section 4.3: extensions ---
if os.path.exists("report/table_v1_3_0_origin_audit.csv"):
    a130 = pd.read_csv("report/table_v1_3_0_origin_audit.csv").iloc[0]
    put("4.3_v1_3_0", "origin_rows", a130["final_rows"])
    put("4.3_v1_3_0", "origin_scenarios", a130["final_scenarios"])
if os.path.exists("data/processed/sdc_v1_3_1_paths.parquet"):
    sd = pd.read_parquet("data/processed/sdc_v1_3_1_paths.parquet")
    put("4.3_sdc", "rows_scenarios", len(sd))
    put("4.3_sdc", "rows_with_valid_path", int(sd["has_valid_path"].sum()))
    put("4.3_sdc", "forecast_agent", "SDC")
numbers = pd.DataFrame({"key": list(P.keys()), "value": list(P.values())})
numbers.to_csv("report/paper_numbers_A.csv", index=False)
with open("report/paper_numbers_A.json", "w", encoding="utf-8") as f:
    json.dump(P, f, indent=2, default=str)
pd.set_option("display.max_rows", 300)
display(numbers)
print("Saved report/paper_numbers_A.csv and report/paper_numbers_A.json")

,key,value
0,3.1_sampling.shards,20.0
1,3.1_sampling.scenarios_scanned,9892.0
2,3.1_sampling.scenarios_91_steps,9892.0
3,3.1_sampling.eligible_vehicle_tracks,157297.0
4,3.1_sampling.selected_tracks,1200.0
5,3.1_sampling.selected_scenarios,1073.0
6,3.1_sampling.vehicles_per_selected_scenario_mean,1.118
7,3.1_sampling.vehicles_per_selected_scenario_max,4.0
8,3.1_sampling.scenarios_with_1_selected_vehicle,965.0
9,3.1_data.rows,109200


Saved report/paper_numbers_A.csv and report/paper_numbers_A.json


## Lỗi thường gặp và cách xử lý (đọc khi thấy chữ đỏ)

| Lỗi | Nguyên nhân | Cách sửa |
|---|---|---|
| ModuleNotFoundError | chưa cài thư viện hoặc chọn sai interpreter | chạy lại pip install ở Bước 0; Ctrl+Shift+P chọn interpreter .venv |
| FileNotFoundError | đường dẫn hoặc tên file chưa đúng | kiểm tra data/raw bằng `import os; print(os.listdir('data/raw'))` |
| KeyError: 'ten_cot' | tên cột thật khác tên trong code | in `df.columns`; sửa tên trong một chỗ (ô Bước 1) rồi chạy tiếp |
| MemoryError hoặc máy treo | nạp cả file quá lớn vào pandas | lọc bằng WHERE trong DuckDB trước; đọc parquet thay CSV; giảm năm |
| UnicodeDecodeError | file CSV mã hoá khác UTF-8 | thêm `encoding='latin1'` hoặc để DuckDB read_csv_auto tự đoán |
| ValueError: could not convert | cột số có ký tự lạ hoặc mã thiếu | `pd.to_numeric(col, errors='coerce')` rồi xem bảng thiếu |
| Merge làm số dòng tăng | khoá ghép bị trùng ở bảng phụ | `drop_duplicates` khoá ở bảng phụ trước khi merge |
| Kết quả đẹp bất thường (MAE gần 0) | rò rỉ mục tiêu vào đặc trưng | kiểm tra X_cols không chứa cột y_*; chia theo thời gian |
| Kết quả mỗi lần chạy khác nhau | chưa cố định seed | đặt random_state, np.random.seed(42) |

Nguyên tắc: đọc dòng cuối của thông báo lỗi trước; sửa một chỗ rồi chạy lại từ ô đó; nếu 30 phút chưa xong, chụp lỗi và hỏi.


## Checklist trước khi báo cáo (đánh dấu [x] khi có file bằng chứng)

- [ ] Restart & Run All chạy hết không lỗi; số thứ tự chạy tăng liên tục
- [ ] Lấy mẫu: `table_sampling_v1_2.csv`, câu mô tả quy tắc chọn 1,200 xe (góp ý 5.4)
- [ ] Làm sạch: `table_cleaning_log.csv` có dòng cắt ngưỡng gia tốc / yaw rate
- [ ] Mục tiêu: `table_target_summary.csv`, `table_zero_displacement.csv` (góp ý 3.3)
- [ ] Bảng chính một thời điểm: `feat_core_origin10.parquet`, `table_core_origin_audit.csv`, `table_core_origin_motion.csv` (góp ý 1a, 3.3, 5.5)
- [ ] RQ1: `table_rq1_density.csv`, `table_rq1_turning.csv`, `table_rq1_stats.csv`, `fig_rq1_density_turning.png` + caption (góp ý 4.3, 4.7)
- [ ] Đèn tín hiệu: `feat_v1_3_0_origin10.parquet`, `feat_v1_3_0_traffic_origin10.parquet` (góp ý 1a, 1b)
- [ ] SDC: `table_sdc_feature_timing.csv`, `sdc_v1_3_1_paths_noroute.parquet`; đã đọc tài liệu WOMD về `on_route` (góp ý 5.3)
- [ ] `paper_numbers_A.csv`: mục 3.1, 3.2, 4.1, 5.1 của bài đã chép số từ file này
- [ ] Manifest bàn giao; Notebook C không còn ghi `feat.parquet` / `manifest.json`
- [ ] Audit Log cá nhân có đủ prompt của tuần


## Mẫu báo cáo tiến độ (slot 3 thứ Tư và thứ Bảy (12:30 đến 14:45), 7:00 đến 9:15)

Dán vào kênh nhóm trước 12:00 ngày báo cáo, mỗi người một phần. Năm mục, mỗi mục hai đến ba dòng:

1. **Đã xong** từ lần trước: tên file, số dòng, số bảng, số hình (ví dụ: `feat.parquet` 138.204 dòng; 4 bảng RQ1; 7 hình).
2. **Con số chính mới có**: ví dụ MAE naive 6,8; MAE LightGBM 4,7 (5 seed, sd 0,1).
3. **Vướng mắc và đã thử gì**: lỗi cụ thể, ảnh chụp, hai cách đã thử.
4. **Sẽ làm đến lần sau, ai làm**: theo Bảng 13 của đề cương.
5. **Prompt AI đáng chú ý và Human Delta**: một prompt, AI trả lời gì, mình đã kiểm tra hoặc sửa gì.

**Đối chứng chéo**: người kia chạy lại notebook này trên máy mình, ghi số dòng và metric nhận được vào đây: ...............................